# **CGA.py**

In [3]:
import torch
from torch import nn
from einops.layers.torch import Rearrange


class SpatialAttention(nn.Module):
    def __init__(self):
        super(SpatialAttention, self).__init__()
        self.sa = nn.Conv2d(2, 1, 7, padding=3, padding_mode='reflect' ,bias=True)

    def forward(self, x):
        x_avg = torch.mean(x, dim=1, keepdim=True)
        x_max, _ = torch.max(x, dim=1, keepdim=True)
        x2 = torch.concat([x_avg, x_max], dim=1)
        sattn = self.sa(x2)
        return sattn


class ChannelAttention(nn.Module):
    def __init__(self, dim, reduction = 8):
        super(ChannelAttention, self).__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.ca = nn.Sequential(
            nn.Conv2d(dim, dim // reduction, 1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // reduction, dim, 1, padding=0, bias=True),
        )

    def forward(self, x):
        x_gap = self.gap(x)
        cattn = self.ca(x_gap)
        return cattn

  
class PixelAttention(nn.Module):
    def __init__(self, dim):
        super(PixelAttention, self).__init__()
        self.pa2 = nn.Conv2d(2 * dim, dim, 7, padding=3, padding_mode='reflect' ,groups=dim, bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, pattn1):
        B, C, H, W = x.shape
        x = x.unsqueeze(dim=2) # B, C, 1, H, W
        pattn1 = pattn1.unsqueeze(dim=2) # B, C, 1, H, W
        x2 = torch.cat([x, pattn1], dim=2) # B, C, 2, H, W
        x2 = Rearrange('b c t h w -> b (c t) h w')(x2)
        pattn2 = self.pa2(x2)
        pattn2 = self.sigmoid(pattn2)
        return pattn2

# **DEConv.py**

In [4]:
import math
import torch
from torch import nn
from einops.layers.torch import Rearrange


class Conv2d_cd(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=1, dilation=1, groups=1, bias=False, theta=1.0):

        super(Conv2d_cd, self).__init__() 
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)
        self.theta = theta
    
    def get_weight(self):
        conv_weight = self.conv.weight
        conv_shape = conv_weight.shape
        conv_weight = Rearrange('c_in c_out k1 k2 -> c_in c_out (k1 k2)')(conv_weight)
        conv_weight_cd = torch.cuda.FloatTensor(conv_shape[0], conv_shape[1], 3 * 3).fill_(0)
        conv_weight_cd[:, :, :] = conv_weight[:, :, :]
        conv_weight_cd[:, :, 4] = conv_weight[:, :, 4] - conv_weight[:, :, :].sum(2)
        conv_weight_cd = Rearrange('c_in c_out (k1 k2) -> c_in c_out k1 k2', k1=conv_shape[2], k2=conv_shape[3])(conv_weight_cd)
        return conv_weight_cd, self.conv.bias


class Conv2d_ad(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=1, dilation=1, groups=1, bias=False, theta=1.0):

        super(Conv2d_ad, self).__init__() 
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)
        self.theta = theta
    
    def get_weight(self):
        conv_weight = self.conv.weight
        conv_shape = conv_weight.shape
        conv_weight = Rearrange('c_in c_out k1 k2 -> c_in c_out (k1 k2)')(conv_weight)
        conv_weight_ad = conv_weight - self.theta * conv_weight[:, :, [3, 0, 1, 6, 4, 2, 7, 8, 5]]
        conv_weight_ad = Rearrange('c_in c_out (k1 k2) -> c_in c_out k1 k2', k1=conv_shape[2], k2=conv_shape[3])(conv_weight_ad)
        return conv_weight_ad, self.conv.bias


class Conv2d_rd(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=2, dilation=1, groups=1, bias=False, theta=1.0):

        super(Conv2d_rd, self).__init__() 
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)
        self.theta = theta

    def forward(self, x):

        if math.fabs(self.theta - 0.0) < 1e-8:
            out_normal = self.conv(x)
            return out_normal 
        else:
            conv_weight = self.conv.weight
            conv_shape = conv_weight.shape
            if conv_weight.is_cuda:
                conv_weight_rd = torch.cuda.FloatTensor(conv_shape[0], conv_shape[1], 5 * 5).fill_(0)
            else:
                conv_weight_rd = torch.zeros(conv_shape[0], conv_shape[1], 5 * 5)
            conv_weight = Rearrange('c_in c_out k1 k2 -> c_in c_out (k1 k2)')(conv_weight)
            conv_weight_rd[:, :, [0, 2, 4, 10, 14, 20, 22, 24]] = conv_weight[:, :, 1:]
            conv_weight_rd[:, :, [6, 7, 8, 11, 13, 16, 17, 18]] = -conv_weight[:, :, 1:] * self.theta
            conv_weight_rd[:, :, 12] = conv_weight[:, :, 0] * (1 - self.theta)
            conv_weight_rd = conv_weight_rd.view(conv_shape[0], conv_shape[1], 5, 5)
            out_diff = nn.functional.conv2d(input=x, weight=conv_weight_rd, bias=self.conv.bias, stride=self.conv.stride, padding=self.conv.padding, groups=self.conv.groups)

            return out_diff


class Conv2d_hd(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=1, dilation=1, groups=1, bias=False, theta=1.0):

        super(Conv2d_hd, self).__init__() 
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)

    def get_weight(self):
        conv_weight = self.conv.weight
        conv_shape = conv_weight.shape
        conv_weight_hd = torch.cuda.FloatTensor(conv_shape[0], conv_shape[1], 3 * 3).fill_(0)
        conv_weight_hd[:, :, [0, 3, 6]] = conv_weight[:, :, :]
        conv_weight_hd[:, :, [2, 5, 8]] = -conv_weight[:, :, :]
        conv_weight_hd = Rearrange('c_in c_out (k1 k2) -> c_in c_out k1 k2', k1=conv_shape[2], k2=conv_shape[2])(conv_weight_hd)
        return conv_weight_hd, self.conv.bias


class Conv2d_vd(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=1, dilation=1, groups=1, bias=False):

        super(Conv2d_vd, self).__init__() 
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)
    
    def get_weight(self):
        conv_weight = self.conv.weight
        conv_shape = conv_weight.shape
        conv_weight_vd = torch.cuda.FloatTensor(conv_shape[0], conv_shape[1], 3 * 3).fill_(0)
        conv_weight_vd[:, :, [0, 1, 2]] = conv_weight[:, :, :]
        conv_weight_vd[:, :, [6, 7, 8]] = -conv_weight[:, :, :]
        conv_weight_vd = Rearrange('c_in c_out (k1 k2) -> c_in c_out k1 k2', k1=conv_shape[2], k2=conv_shape[2])(conv_weight_vd)
        return conv_weight_vd, self.conv.bias


class DEConv(nn.Module):
    def __init__(self, dim):
        super(DEConv, self).__init__() 
        self.conv1_1 = Conv2d_cd(dim, dim, 3, bias=True)
        self.conv1_2 = Conv2d_hd(dim, dim, 3, bias=True)
        self.conv1_3 = Conv2d_vd(dim, dim, 3, bias=True)
        self.conv1_4 = Conv2d_ad(dim, dim, 3, bias=True)
        self.conv1_5 = nn.Conv2d(dim, dim, 3, padding=1, bias=True)
    

    def forward(self, x):
        w1, b1 = self.conv1_1.get_weight()
        w2, b2 = self.conv1_2.get_weight()
        w3, b3 = self.conv1_3.get_weight()
        w4, b4 = self.conv1_4.get_weight()
        w5, b5 = self.conv1_5.weight, self.conv1_5.bias

        w = w1 + w2 + w3 + w4 + w5
        b = b1 + b2 + b3 + b4 + b5
        res = nn.functional.conv2d(input=x, weight=w, bias=b, stride=1, padding=1, groups=1)

        return res

# **DEABlock_train.py**


In [5]:
from torch import nn

# from .deconv import DEConv
# from .cga import SpatialAttention, ChannelAttention, PixelAttention


class DEABlockTrain(nn.Module):
    def __init__(self, conv, dim, kernel_size, reduction=8):
        super(DEABlockTrain, self).__init__()
        self.conv1 = DEConv(dim)
        self.act1 = nn.ReLU(inplace=True)
        self.conv2 = conv(dim, dim, kernel_size, bias=True)
        self.sa = SpatialAttention()
        self.ca = ChannelAttention(dim, reduction)
        self.pa = PixelAttention(dim)

    def forward(self, x):
        res = self.conv1(x)
        res = self.act1(res)
        res = res + x
        res = self.conv2(res)
        cattn = self.ca(res)
        sattn = self.sa(res)
        pattn1 = sattn + cattn
        pattn2 = self.pa(res, pattn1)
        res = res * pattn2
        res = res + x
        return res


class DEBlockTrain(nn.Module):
    def __init__(self, conv, dim, kernel_size):
        super(DEBlockTrain, self).__init__()
        self.conv1 = DEConv(dim)
        self.act1 = nn.ReLU(inplace=True)
        self.conv2 = conv(dim, dim, kernel_size, bias=True)

    def forward(self, x):
        res = self.conv1(x)
        res = self.act1(res)
        res = res + x
        res = self.conv2(res)
        res = res + x
        return res

# **Fusion.py**

In [6]:
from torch import nn

# from .cga import SpatialAttention, ChannelAttention, PixelAttention


class CGAFusion(nn.Module):
    def __init__(self, dim, reduction=8):
        super(CGAFusion, self).__init__()
        self.sa = SpatialAttention()
        self.ca = ChannelAttention(dim, reduction)
        self.pa = PixelAttention(dim)
        self.conv = nn.Conv2d(dim, dim, 1, bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, y):
        initial = x + y
        cattn = self.ca(initial)
        sattn = self.sa(initial)
        pattn1 = sattn + cattn
        pattn2 = self.sigmoid(self.pa(initial, pattn1))
        result = initial + pattn2 * x + (1 - pattn2) * y
        result = self.conv(result)
        return result

# **logger.py**

In [7]:
from matplotlib import pyplot as plt
import numpy as np
import os


def plot_loss_log(loss_log, epoch, loss_dir):
    axis = np.linspace(1, epoch, epoch)
    for key in loss_log.keys():
        label = '{} Loss'.format(key)
        fig = plt.figure()
        plt.title(label)
        plt.plot(axis, np.array(loss_log[key]))
        plt.legend()
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.savefig(os.path.join(loss_dir, 'loss_{}.pdf'.format(key)))
        plt.close(fig)


def plot_psnr_log(psnr_log, epoch, psnr_dir):
    axis = np.linspace(1, epoch, epoch)
    label = 'PSNR'
    fig = plt.figure()
    plt.title(label)
    plt.plot(axis, np.array(psnr_log))
    plt.legend()
    plt.xlabel('Epochs')
    plt.ylabel('PSNR')
    plt.grid(True)
    plt.savefig(os.path.join(psnr_dir, 'psnr.pdf'))
    plt.close(fig)

# *metric.py***

In [8]:
import math
from math import exp
import numpy as np
import torch
import torch.nn.functional as F
from torch.autograd import Variable


def gaussian(window_size, sigma):
    gauss = torch.Tensor([exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])
    return gauss / gauss.sum()


def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())
    return window


def _ssim(img1, img2, window, window_size, channel, size_average=True):
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)
    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2
    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map.mean(1).mean(1).mean(1)


def ssim(img1, img2, window_size=11, size_average=True):
    img1 = torch.clamp(img1, min=0, max=1)
    img2 = torch.clamp(img2, min=0, max=1)
    (_, channel, _, _) = img1.size()
    window = create_window(window_size, channel)
    if img1.is_cuda:
        window = window.cuda(img1.get_device())
    window = window.type_as(img1)
    return _ssim(img1, img2, window, window_size, channel, size_average)


def psnr(pred, gt):
    pred = pred.clamp(0, 1).cpu().numpy()
    gt = gt.clamp(0, 1).cpu().numpy()
    imdff = pred - gt
    rmse = math.sqrt(np.mean(imdff ** 2))
    if rmse == 0:
        return 100
    return 20 * math.log10(1.0 / rmse)

# **options.py**

In [9]:
import sys

# Remove the notebook argument if present
if '-f' in sys.argv:
    sys.argv = sys.argv[:1]

# Rest of your argument parsing code here


In [10]:
# import os,argparse
# import json


# parser = argparse.ArgumentParser()

# parser.add_argument('--exp_dir', type=str, default='../experiment')
# parser.add_argument('--dataset', type=str, default='ITS')
# parser.add_argument('--val_dataset_dir', type=str)
# parser.add_argument('--model_name', type=str, default='DEA-Net', help='experiment name')
# parser.add_argument('--saved_infer_dir', type=str, default='saved_infer_dir')

# # only need for evaluation
# parser.add_argument('--pre_trained_model', type=str, default='null', help='path of pre trained model for resume training')
# parser.add_argument('--save_infer_results', action='store_true', default=False, help='save the infer results during validation')
# opt=parser.parse_args()

# opt.val_dataset_dir = os.path.join('../dataset/', opt.dataset, 'test')
# exp_dataset_dir = os.path.join(opt.exp_dir, opt.dataset)
# exp_model_dir = os.path.join(exp_dataset_dir, opt.model_name)

# if not os.path.exists(opt.exp_dir):
#     os.mkdir(opt.exp_dir)
 
# if not os.path.exists(exp_dataset_dir):
#     os.mkdir(exp_dataset_dir)

# opt.saved_infer_dir = os.path.join(exp_model_dir, opt.pre_trained_model.split('.pth')[0])
# if not os.path.exists(exp_model_dir):
#     os.mkdir(exp_model_dir)
#     os.mkdir(opt.saved_infer_dir)
# if not os.path.exists(opt.saved_infer_dir):
#     os.mkdir(opt.saved_infer_dir)

# with open(os.path.join(exp_model_dir, 'args.txt'), 'w') as f:
#     json.dump(opt.__dict__, f, indent=2)

In [9]:
# import os

# # File path to delete
# file_path = "/kaggle/working/options_train.py"

# # Remove the file
# if os.path.exists(file_path):
#     os.remove(file_path)
#     print(f"{file_path} has been deleted.")
# else:
#     print(f"{file_path} does not exist.")


In [10]:
# import shutil

# # Source file path
# source_path = "/kaggle/input/optionss/options.py"

# # Destination file path with new name
# destination_path = "/kaggle/working/"

# # Copy and rename the file
# shutil.copy(source_path, destination_path)

# print(f"File copied and renamed to {destination_path}")


In [11]:
# import shutil

# # Source file path
# source_path = "/kaggle/input/options-trainn/options_train.py"

# # Destination file path with new name
# destination_path = "/kaggle/working/"

# # Copy and rename the file
# shutil.copy(source_path, destination_path)

# print(f"File copied and renamed to {destination_path}")


# actual options_train.py

In [11]:
import torch
import os
import json

class TrainOptions:
    def __init__(self):
        # Training configuration
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.epochs = 10
        self.iters_per_epoch = 5
        self.finer_eval_step = 4
        self.bs = 16
        self.start_lr = 0.0004
        self.end_lr = 0.000001
        self.no_lr_sche = False
        self.use_warm_up = False

        # Loss weights
        self.w_loss_L1 = 1.0
        self.w_loss_CR = 0.1

        # Experiment setup
        self.exp_dir = './experiment'
        self.model_name = 'MDCTDN'
        self.dataset = 'ITS'

        # Resume training
        self.resume = False
        self.pre_trained_model = 'null'

        # Derived paths
        self.dataset_dir = os.path.join(self.exp_dir, self.dataset)
        self.model_dir = os.path.join(self.dataset_dir, self.model_name)
        self.saved_model_dir = os.path.join(self.model_dir, 'saved_model')
        self.saved_data_dir = os.path.join(self.model_dir, 'saved_data')
        self.saved_plot_dir = os.path.join(self.model_dir, 'saved_plot')
        self.saved_infer_dir = os.path.join(self.model_dir, 'saved_infer')

        # Create necessary folders
        self._create_directories()

    # def _create_directories(self):
    #     # Make directories if they don't exist
    #     os.makedirs(self.exp_dir, exist_ok=True)
    #     os.makedirs(self.dataset_dir, exist_ok=True)

    #     if os.path.exists(self.model_dir):
    #         print(f"{self.model_dir} has already existed!")
    #         exit()
    #     else:
    #         os.makedirs(self.model_dir)
    #         os.makedirs(self.saved_model_dir)
    #         os.makedirs(self.saved_data_dir)
    #         os.makedirs(self.saved_plot_dir)
    #         os.makedirs(self.saved_infer_dir)
    def _create_directories(self):
    # Create base directories if not present
        os.makedirs(self.exp_dir, exist_ok=True)
        os.makedirs(self.dataset_dir, exist_ok=True)
        os.makedirs(self.model_dir, exist_ok=True)
        os.makedirs(self.saved_model_dir, exist_ok=True)
        os.makedirs(self.saved_data_dir, exist_ok=True)
        os.makedirs(self.saved_plot_dir, exist_ok=True)
        os.makedirs(self.saved_infer_dir, exist_ok=True)

    def save_args(self):
        # Save the full configuration to args.txt
        with open(os.path.join(self.model_dir, 'args.txt'), 'w') as f:
            json.dump(self.__dict__, f, indent=2)

    def __str__(self):
        return f"TrainOptions(device={self.device}, epochs={self.epochs}, iters_per_epoch={self.iters_per_epoch}, " \
               f"bs={self.bs}, start_lr={self.start_lr}, end_lr={self.end_lr}, " \
               f"exp_dir={self.exp_dir}, model_name={self.model_name}, dataset={self.dataset})"


# Instantiate and use
opt = TrainOptions()
print(opt)
opt.save_args()


TrainOptions(device=cuda, epochs=10, iters_per_epoch=5, bs=16, start_lr=0.0004, end_lr=1e-06, exp_dir=./experiment, model_name=MDCTDN, dataset=ITS)


# **options_train.py**

In [13]:
import torch
import os
import json

class TrainOptions:
    def __init__(self):
        # Default values
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.epochs = 5
        self.iters_per_epoch = 1
        self.finer_eval_step = 400000
        self.bs = 1  # batch size
        self.start_lr = 0.0004
        self.end_lr = 0.000001
        self.no_lr_sche = False
        self.use_warm_up = False
        self.exp_dir = './experiment'  # Use current directory to avoid permission errors
        self.model_name = 'MDCTDN'
        self.dataset = 'ITS'

        # Resume training
        self.resume = False
        self.pre_trained_model = 'null'

        # Directories
        self.dataset_dir = os.path.join(self.exp_dir, self.dataset)
        self.model_dir = os.path.join(self.dataset_dir, self.model_name)

        # Create necessary directories
        self._create_directories()

    def _create_directories(self):
        # Use os.makedirs with exist_ok=True to avoid errors
        os.makedirs(self.exp_dir, exist_ok=True)
        os.makedirs(self.dataset_dir, exist_ok=True)
        os.makedirs(self.model_dir, exist_ok=True)

        self.saved_model_dir = os.path.join(self.model_dir, 'saved_model')
        self.saved_data_dir = os.path.join(self.model_dir, 'saved_data')
        self.saved_plot_dir = os.path.join(self.model_dir, 'saved_plot')
        self.saved_infer_dir = os.path.join(self.model_dir, 'saved_infer')

        os.makedirs(self.saved_model_dir, exist_ok=True)
        os.makedirs(self.saved_data_dir, exist_ok=True)
        os.makedirs(self.saved_plot_dir, exist_ok=True)
        os.makedirs(self.saved_infer_dir, exist_ok=True)

    def save_args(self):
        # Save the options to a JSON file
        with open(os.path.join(self.model_dir, 'args.txt'), 'w') as f:
            json.dump(self.__dict__, f, indent=2)

    def __str__(self):
        return f"TrainOptions(device={self.device}, epochs={self.epochs}, iters_per_epoch={self.iters_per_epoch}, " \
               f"bs={self.bs}, start_lr={self.start_lr}, end_lr={self.end_lr}, " \
               f"exp_dir={self.exp_dir}, model_name={self.model_name}, dataset={self.dataset})"


# Instantiate the class and use the options
opt = TrainOptions()

# Print the options to verify
print(opt)

# Save arguments to 'args.txt'
opt.save_args()


TrainOptions(device=cuda, epochs=5, iters_per_epoch=1, bs=1, start_lr=0.0004, end_lr=1e-06, exp_dir=./experiment, model_name=MDCTDN, dataset=ITS)


In [14]:
# import torch,os,sys,torchvision,argparse
# import torch,warnings
# import json

# # warnings.filterwarnings('ignore')

# parser = argparse.ArgumentParser()

# parser.add_argument('--device', type=str,default='Automatic detection')

# parser.add_argument('--epochs', type=int,default=100)
# parser.add_argument('--iters_per_epoch', type=int,default=5000)
# parser.add_argument('--finer_eval_step', type=int,default=400000)
# parser.add_argument('--bs', type=int,default=16,help='batch size')
# parser.add_argument('--start_lr', default=0.0004, type=float, help='start learning rate')
# parser.add_argument('--end_lr', default=0.000001, type=float, help='end learning rate')
# parser.add_argument('--no_lr_sche', action='store_true',help='no lr cos schedule')
# parser.add_argument('--use_warm_up', type=bool, default=False, help='using warm up in learning rate')

# # parser.add_argument('--w_loss_L1', default=1., type=float, help='weight of loss L1')
# # parser.add_argument('--w_loss_CR', default=0.1, type=float, help='weight of loss CR')

# parser.add_argument('--exp_dir', type=str, default='../experiment')
# parser.add_argument('--model_name', type=str, default='MDCTDN')
# parser.add_argument('--saved_model_dir', type=str, default='saved_model')
# parser.add_argument('--saved_data_dir', type=str, default='saved_data')
# parser.add_argument('--saved_plot_dir', type=str, default='saved_plot')
# parser.add_argument('--saved_infer_dir', type=str, default='saved_infer_dir')

# parser.add_argument('--dataset', type=str, default='ITS')

# # only need for resume
# parser.add_argument('--resume', type=bool,default=False)
# parser.add_argument('--pre_trained_model', type=str,default='null')

# opt=parser.parse_args()
# opt.device='cuda' if torch.cuda.is_available() else 'cpu'

# dataset_dir = os.path.join(opt.exp_dir, opt.dataset)
# model_dir = os.path.join(dataset_dir, opt.model_name)

# if not os.path.exists(opt.exp_dir):
#     os.mkdir(opt.exp_dir)
# if not os.path.exists(dataset_dir):
#     os.mkdir(dataset_dir)
# if not os.path.exists(model_dir):
#     os.mkdir(model_dir)
#     opt.saved_model_dir = os.path.join(model_dir, 'saved_model')
#     opt.saved_data_dir = os.path.join(model_dir, 'saved_data')
#     opt.saved_plot_dir = os.path.join(model_dir, 'saved_plot')
#     opt.saved_infer_dir = os.path.join(model_dir, 'saved_infer')
#     os.mkdir(opt.saved_model_dir)
#     os.mkdir(opt.saved_data_dir)
#     os.mkdir(opt.saved_plot_dir)
#     os.mkdir(opt.saved_infer_dir)
# else:
#     print(f'{model_dir} has already existed!')
#     exit()

# print(opt)
# print('model_dir:', model_dir)

# with open(os.path.join(model_dir, 'args.txt'), 'w') as f:
#     json.dump(opt.__dict__, f, indent=2)

In [15]:
# import shutil

# # Source file path
# source_path = "/kaggle/input/opt-train/options-train-py (2).ipynb"

# # Destination file path with new name
# destination_path = "/kaggle/working/options_train.py"

# # Copy and rename the file
# shutil.copy(source_path, destination_path)

# print(f"File copied and renamed to {destination_path}")


# **Prompt_loader**

In [15]:
# import os
# import random
# import copy
# from PIL import Image
# import numpy as np

# from torch.utils.data import Dataset
# from torchvision.transforms import ToPILImage, Compose, RandomCrop, ToTensor
# import torch

# from utils.image_utils import random_augmentation, crop_img
# from utils.degradation_utils import Degradation

    
# class PromptTrainDataset(Dataset):
#     def __init__(self, args):
#         super(PromptTrainDataset, self).__init__()
#         self.args = args
#         self.rs_ids = []
#         self.hazy_ids = []
#         self.D = Degradation(args)
#         self.de_temp = 0
#         self.de_type = self.args.de_type
#         print(self.de_type)

#         self.de_dict = {'denoise_15': 0, 'denoise_25': 1, 'denoise_50': 2, 'derain': 3, 'dehaze': 4, 'deblur' : 5}

#         self._init_ids()
#         self._merge_ids()

#         self.crop_transform = Compose([
#             ToPILImage(),
#             RandomCrop(args.patch_size),
#         ])

#         self.toTensor = ToTensor()

#     def _init_ids(self):
#         if 'denoise_15' in self.de_type or 'denoise_25' in self.de_type or 'denoise_50' in self.de_type:
#             self._init_clean_ids()
#         if 'derain' in self.de_type:
#             self._init_rs_ids()
#         if 'dehaze' in self.de_type:
#             self._init_hazy_ids()

#         random.shuffle(self.de_type)

#     def _init_clean_ids(self):
#         ref_file = self.args.data_file_dir + "noisy/denoise_airnet.txt"
#         temp_ids = []
#         temp_ids+= [id_.strip() for id_ in open(ref_file)]
#         clean_ids = []
#         name_list = os.listdir(self.args.denoise_dir)
#         clean_ids += [self.args.denoise_dir + id_ for id_ in name_list if id_.strip() in temp_ids]

#         if 'denoise_15' in self.de_type:
#             self.s15_ids = [{"clean_id": x,"de_type":0} for x in clean_ids]
#             self.s15_ids = self.s15_ids * 3
#             random.shuffle(self.s15_ids)
#             self.s15_counter = 0
#         if 'denoise_25' in self.de_type:
#             self.s25_ids = [{"clean_id": x,"de_type":1} for x in clean_ids]
#             self.s25_ids = self.s25_ids * 3
#             random.shuffle(self.s25_ids)
#             self.s25_counter = 0
#         if 'denoise_50' in self.de_type:
#             self.s50_ids = [{"clean_id": x,"de_type":2} for x in clean_ids]
#             self.s50_ids = self.s50_ids * 3
#             random.shuffle(self.s50_ids)
#             self.s50_counter = 0

#         self.num_clean = len(clean_ids)
#         print("Total Denoise Ids : {}".format(self.num_clean))

#     def _init_hazy_ids(self):
#         temp_ids = []
#         hazy = self.args.data_file_dir + "hazy/hazy_outside.txt"
#         temp_ids+= [self.args.dehaze_dir + id_.strip() for id_ in open(hazy)]
#         self.hazy_ids = [{"clean_id" : x,"de_type":4} for x in temp_ids]

#         self.hazy_counter = 0
        
#         self.num_hazy = len(self.hazy_ids)
#         print("Total Hazy Ids : {}".format(self.num_hazy))

#     def _init_rs_ids(self):
#         temp_ids = []
#         rs = self.args.data_file_dir + "rainy/rainTrain.txt"
#         temp_ids+= [self.args.derain_dir + id_.strip() for id_ in open(rs)]
#         self.rs_ids = [{"clean_id":x,"de_type":3} for x in temp_ids]
#         self.rs_ids = self.rs_ids * 120

#         self.rl_counter = 0
#         self.num_rl = len(self.rs_ids)
#         print("Total Rainy Ids : {}".format(self.num_rl))
    

#     def _crop_patch(self, img_1, img_2):
#         H = img_1.shape[0]
#         W = img_1.shape[1]
#         ind_H = random.randint(0, H - self.args.patch_size)
#         ind_W = random.randint(0, W - self.args.patch_size)

#         patch_1 = img_1[ind_H:ind_H + self.args.patch_size, ind_W:ind_W + self.args.patch_size]
#         patch_2 = img_2[ind_H:ind_H + self.args.patch_size, ind_W:ind_W + self.args.patch_size]

#         return patch_1, patch_2

#     def _get_gt_name(self, rainy_name):
#         gt_name = rainy_name.split("rainy")[0] + 'gt/norain-' + rainy_name.split('rain-')[-1]
#         return gt_name

#     def _get_nonhazy_name(self, hazy_name):
#         dir_name = hazy_name.split("synthetic")[0] + 'original/'
#         name = hazy_name.split('/')[-1].split('_')[0]
#         suffix = '.' + hazy_name.split('.')[-1]
#         nonhazy_name = dir_name + name + suffix
#         return nonhazy_name

#     def _merge_ids(self):
#         self.sample_ids = []
#         if "denoise_15" in self.de_type:
#             self.sample_ids += self.s15_ids
#             self.sample_ids += self.s25_ids
#             self.sample_ids += self.s50_ids
#         if "derain" in self.de_type:
#             self.sample_ids+= self.rs_ids
        
#         if "dehaze" in self.de_type:
#             self.sample_ids+= self.hazy_ids
#         print(len(self.sample_ids))

#     def __getitem__(self, idx):
#         sample = self.sample_ids[idx]
#         de_id = sample["de_type"]

#         if de_id < 3:
#             if de_id == 0:
#                 clean_id = sample["clean_id"]
#             elif de_id == 1:
#                 clean_id = sample["clean_id"]
#             elif de_id == 2:
#                 clean_id = sample["clean_id"]

#             clean_img = crop_img(np.array(Image.open(clean_id).convert('RGB')), base=16)
#             clean_patch = self.crop_transform(clean_img)
#             clean_patch= np.array(clean_patch)

#             clean_name = clean_id.split("/")[-1].split('.')[0]

#             clean_patch = random_augmentation(clean_patch)[0]

#             degrad_patch = self.D.single_degrade(clean_patch, de_id)
#         else:
#             if de_id == 3:
#                 # Rain Streak Removal
#                 degrad_img = crop_img(np.array(Image.open(sample["clean_id"]).convert('RGB')), base=16)
#                 clean_name = self._get_gt_name(sample["clean_id"])
#                 clean_img = crop_img(np.array(Image.open(clean_name).convert('RGB')), base=16)
#             elif de_id == 4:
#                 # Dehazing with SOTS outdoor training set
#                 degrad_img = crop_img(np.array(Image.open(sample["clean_id"]).convert('RGB')), base=16)
#                 clean_name = self._get_nonhazy_name(sample["clean_id"])
#                 clean_img = crop_img(np.array(Image.open(clean_name).convert('RGB')), base=16)

#             degrad_patch, clean_patch = random_augmentation(*self._crop_patch(degrad_img, clean_img))

#         clean_patch = self.toTensor(clean_patch)
#         degrad_patch = self.toTensor(degrad_patch)


#         return [clean_name, de_id], degrad_patch, clean_patch

#     def __len__(self):
#         return len(self.sample_ids)


# class DenoiseTestDataset(Dataset):
#     def __init__(self, args):
#         super(DenoiseTestDataset, self).__init__()
#         self.args = args
#         self.clean_ids = []
#         self.sigma = 15

#         self._init_clean_ids()

#         self.toTensor = ToTensor()

#     def _init_clean_ids(self):
#         name_list = os.listdir(self.args.denoise_path)
#         self.clean_ids += [self.args.denoise_path + id_ for id_ in name_list]

#         self.num_clean = len(self.clean_ids)

#     def _add_gaussian_noise(self, clean_patch):
#         noise = np.random.randn(*clean_patch.shape)
#         noisy_patch = np.clip(clean_patch + noise * self.sigma, 0, 255).astype(np.uint8)
#         return noisy_patch, clean_patch

#     def set_sigma(self, sigma):
#         self.sigma = sigma

#     def __getitem__(self, clean_id):
#         clean_img = crop_img(np.array(Image.open(self.clean_ids[clean_id]).convert('RGB')), base=16)
#         clean_name = self.clean_ids[clean_id].split("/")[-1].split('.')[0]

#         noisy_img, _ = self._add_gaussian_noise(clean_img)
#         clean_img, noisy_img = self.toTensor(clean_img), self.toTensor(noisy_img)

#         return [clean_name], noisy_img, clean_img
#     def tile_degrad(input_,tile=128,tile_overlap =0):
#         sigma_dict = {0:0,1:15,2:25,3:50}
#         b, c, h, w = input_.shape
#         tile = min(tile, h, w)
#         assert tile % 8 == 0, "tile size should be multiple of 8"

#         stride = tile - tile_overlap
#         h_idx_list = list(range(0, h-tile, stride)) + [h-tile]
#         w_idx_list = list(range(0, w-tile, stride)) + [w-tile]
#         E = torch.zeros(b, c, h, w).type_as(input_)
#         W = torch.zeros_like(E)
#         s = 0
#         for h_idx in h_idx_list:
#             for w_idx in w_idx_list:
#                 in_patch = input_[..., h_idx:h_idx+tile, w_idx:w_idx+tile]
#                 out_patch = in_patch
#                 # out_patch = model(in_patch)
#                 out_patch_mask = torch.ones_like(in_patch)

#                 E[..., h_idx:(h_idx+tile), w_idx:(w_idx+tile)].add_(out_patch)
#                 W[..., h_idx:(h_idx+tile), w_idx:(w_idx+tile)].add_(out_patch_mask)
#         # restored = E.div_(W)

#         restored = torch.clamp(restored, 0, 1)
#         return restored
#     def __len__(self):
#         return self.num_clean


# class DerainDehazeDataset(Dataset):
#     def __init__(self, args, task="derain",addnoise = False,sigma = None):
#         super(DerainDehazeDataset, self).__init__()
#         self.ids = []
#         self.task_idx = 0
#         self.args = args

#         self.task_dict = {'derain': 0, 'dehaze': 1}
#         self.toTensor = ToTensor()
#         self.addnoise = addnoise
#         self.sigma = sigma

#         self.set_dataset(task)
#     def _add_gaussian_noise(self, clean_patch):
#         noise = np.random.randn(*clean_patch.shape)
#         noisy_patch = np.clip(clean_patch + noise * self.sigma, 0, 255).astype(np.uint8)
#         return noisy_patch, clean_patch

#     def _init_input_ids(self):
#         if self.task_idx == 0:
#             self.ids = []
#             name_list = os.listdir(self.args.derain_path + 'input/')
#             # print(name_list)
#             print(self.args.derain_path)
#             self.ids += [self.args.derain_path + 'input/' + id_ for id_ in name_list]
#         elif self.task_idx == 1:
#             self.ids = []
#             name_list = os.listdir(self.args.dehaze_path + 'input/')
#             self.ids += [self.args.dehaze_path + 'input/' + id_ for id_ in name_list]

#         self.length = len(self.ids)

#     def _get_gt_path(self, degraded_name):
#         if self.task_idx == 0:
#             gt_name = degraded_name.replace("input", "target")
#         elif self.task_idx == 1:
#             dir_name = degraded_name.split("input")[0] + 'target/'
#             name = degraded_name.split('/')[-1].split('_')[0] + '.png'
#             gt_name = dir_name + name
#         return gt_name

#     def set_dataset(self, task):
#         self.task_idx = self.task_dict[task]
#         self._init_input_ids()

#     def __getitem__(self, idx):
#         degraded_path = self.ids[idx]
#         clean_path = self._get_gt_path(degraded_path)

#         degraded_img = crop_img(np.array(Image.open(degraded_path).convert('RGB')), base=16)
#         if self.addnoise:
#             degraded_img,_ = self._add_gaussian_noise(degraded_img)
#         clean_img = crop_img(np.array(Image.open(clean_path).convert('RGB')), base=16)

#         clean_img, degraded_img = self.toTensor(clean_img), self.toTensor(degraded_img)
#         degraded_name = degraded_path.split('/')[-1][:-4]

#         return [degraded_name], degraded_img, clean_img

#     def __len__(self):
#         return self.length


# class TestSpecificDataset(Dataset):
#     def __init__(self, args):
#         super(TestSpecificDataset, self).__init__()
#         self.args = args
#         self.degraded_ids = []
#         self._init_clean_ids(args.test_path)

#         self.toTensor = ToTensor()

#     def _init_clean_ids(self, root):
#         extensions = ['jpg', 'JPG', 'png', 'PNG', 'jpeg', 'JPEG', 'bmp', 'BMP']
#         if os.path.isdir(root):
#             name_list = []
#             for image_file in os.listdir(root):
#                 if any([image_file.endswith(ext) for ext in extensions]):
#                     name_list.append(image_file)
#             if len(name_list) == 0:
#                 raise Exception('The input directory does not contain any image files')
#             self.degraded_ids += [root + id_ for id_ in name_list]
#         else:
#             if any([root.endswith(ext) for ext in extensions]):
#                 name_list = [root]
#             else:
#                 raise Exception('Please pass an Image file')
#             self.degraded_ids = name_list
#         print("Total Images : {}".format(name_list))

#         self.num_img = len(self.degraded_ids)

#     def __getitem__(self, idx):
#         degraded_img = crop_img(np.array(Image.open(self.degraded_ids[idx]).convert('RGB')), base=16)
#         name = self.degraded_ids[idx].split('/')[-1][:-4]

#         degraded_img = self.toTensor(degraded_img)

#         return [name], degraded_img

#     def __len__(self):
#         return self.num_img
    

# **data_loader.py**

In [16]:
# !pip uninstall -y torchvision 
# !pip cache purge


In [17]:
# !pip install torchvision==0.16.0

In [18]:
# pip install numpy==1.23.5


In [19]:
# !pip install torch==2.5.1
# !pip install tensorflow==2.17.0
# !pip install fsspec==2024.10.0


In [20]:
# !pip install torch==2.1.0 torchvision==0.16.0 --force-reinstall

In [21]:
import os
import random
import torch.utils.data as data
from PIL import Image
from torchvision.transforms.functional import hflip, rotate, crop
from torchvision.transforms import ToTensor, RandomCrop
from albumentations import RandomCrop




def find_clear_image_name(degraded_name, clear_path):
    base_name = degraded_name.split('_')[0].replace("rain-", "norain-")

    if not base_name.isdigit():
        return None

    possible_files = [
        f"{int(base_name):04d}.png", 
        f"{int(base_name):04d}.jpg"
    ]

    for file in possible_files:
        file_path = os.path.join(clear_path, file)
        if os.path.exists(file_path):
            return file

    return None

class TrainDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(TrainDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path
        self.degraded_image_list = [f for f in os.listdir(degraded_path) if f.split('_')[0].isdigit()][:1]

    # def __getitem__(self, index):
    #     degraded_image_name = self.degraded_image_list[index]
    #     clear_image_name = find_clear_image_name(degraded_image_name, self.clear_path)

    #     if clear_image_name is None:
    #         return None

    #     degraded_image_path = os.path.join(self.degraded_path, degraded_image_name)
    #     clear_image_path = os.path.join(self.clear_path, clear_image_name)

    #     degraded = Image.open(degraded_image_path).convert('RGB')
    #     clear = Image.open(clear_image_path).convert('RGB')

    #     crop_params = RandomCrop.get_params(degraded, (256, 256))
    #     import torch

    #     # crop_params = RandomCrop.get_params(degraded, output_size=(256, 256), generator=torch.Generator().manual_seed(random.randint(0, 2**32 - 1)))

    #     rotate_params = random.randint(0, 3) * 90

    #     degraded = crop(degraded, *crop_params)
    #     clear = crop(clear, *crop_params)

    #     degraded = rotate(degraded, rotate_params)
    #     clear = rotate(clear, rotate_params)

    #     to_tensor = ToTensor()
    #     degraded = to_tensor(degraded)
    #     clear = to_tensor(clear)

    #     return degraded, clear, degraded_image_path


    
    def __getitem__(self, index):
        degraded_image_name = self.degraded_image_list[index]
        clear_image_name = find_clear_image_name(degraded_image_name, self.clear_path)
    
        if clear_image_name is None:
            return None
    
        degraded_image_path = os.path.join(self.degraded_path, degraded_image_name)
        clear_image_path = os.path.join(self.clear_path, clear_image_name)
    
        degraded = Image.open(degraded_image_path).convert('RGB')
        clear = Image.open(clear_image_path).convert('RGB')
    
        degraded_np = np.array(degraded)
        clear_np = np.array(clear)
    
        # Apply RandomCrop correctly
        crop_transform = RandomCrop(height=256, width=256)
        augmented = crop_transform(image=degraded_np, mask=clear_np)
    
        degraded_np = augmented["image"]
        clear_np = augmented["mask"]
    
        # Convert back to PIL for additional transformations
        degraded = Image.fromarray(degraded_np)
        clear = Image.fromarray(clear_np)
    
        # Apply random rotation
        rotate_params = random.randint(0, 3) * 90
        degraded = degraded.rotate(rotate_params)
        clear = clear.rotate(rotate_params)
    
        # Convert to tensors
        to_tensor = ToTensor()
        degraded = to_tensor(degraded)
        clear = to_tensor(clear)
    
        return degraded, clear, degraded_image_path
    def __len__(self):
        return len(self.degraded_image_list)


class TestDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(TestDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path
        self.degraded_image_list = [f for f in sorted(os.listdir(degraded_path)) if f.split('_')[0].isdigit()][:1]

    def __getitem__(self, index):
        degraded_image_name = self.degraded_image_list[index]
        clear_image_name = find_clear_image_name(degraded_image_name, self.clear_path)

        if clear_image_name is None:
            return None

        degraded_image_path = os.path.join(self.degraded_path, degraded_image_name)
        clear_image_path = os.path.join(self.clear_path, clear_image_name)

        degraded = Image.open(degraded_image_path).convert('RGB')
        clear = Image.open(clear_image_path).convert('RGB')

        to_tensor = ToTensor()
        degraded = to_tensor(degraded)
        clear = to_tensor(clear)

        return degraded, clear, degraded_image_path

    def __len__(self):
        return len(self.degraded_image_list)


class ValDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(ValDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path
        self.degraded_image_list = [f for f in sorted(os.listdir(degraded_path)) if f.split('_')[0].isdigit()]

    def __getitem__(self, index):
        degraded_image_name = self.degraded_image_list[index]
        clear_image_name = find_clear_image_name(degraded_image_name, self.clear_path)

        if clear_image_name is None:
            return None

        degraded_image_path = os.path.join(self.degraded_path, degraded_image_name)
        clear_image_path = os.path.join(self.clear_path, clear_image_name)

        degraded = Image.open(degraded_image_path).convert('RGB')
        clear = Image.open(clear_image_path).convert('RGB')

        to_tensor = ToTensor()
        degraded = to_tensor(degraded)
        clear = to_tensor(clear)

        return {'degraded': degraded, 'clear': clear, 'filename': degraded_image_name}

    def __len__(self):
        return len(self.degraded_image_list)


/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.5 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


# data-loader for rain100L

In [1]:
import os
import random
import numpy as np
from PIL import Image
import torch.utils.data as data
from torchvision.transforms import ToTensor
from albumentations import RandomCrop


def get_matching_clear_name(filename):
    return filename  # same name in both `input` and `target`


class TrainDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(TrainDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path

        all_filenames = sorted([f for f in os.listdir(degraded_path) if f.endswith(".png")],
                               key=lambda x: int(os.path.splitext(x)[0]))
        self.degraded_image_list = all_filenames[:80]  # training on first 80

    def __getitem__(self, index):
        degraded_name = self.degraded_image_list[index]
        clear_name = get_matching_clear_name(degraded_name)

        degraded_img = Image.open(os.path.join(self.degraded_path, degraded_name)).convert('RGB')
        clear_img = Image.open(os.path.join(self.clear_path, clear_name)).convert('RGB')

        degraded_np = np.array(degraded_img)
        clear_np = np.array(clear_img)

        # Random crop
        crop = RandomCrop(height=256, width=256)
        augmented = crop(image=degraded_np, mask=clear_np)
        degraded_np = augmented['image']
        clear_np = augmented['mask']

        degraded_img = Image.fromarray(degraded_np)
        clear_img = Image.fromarray(clear_np)

        # Random rotation
        angle = random.choice([0, 90, 180, 270])
        degraded_img = degraded_img.rotate(angle)
        clear_img = clear_img.rotate(angle)

        to_tensor = ToTensor()
        degraded_tensor = to_tensor(degraded_img)
        clear_tensor = to_tensor(clear_img)

        return degraded_tensor, clear_tensor

    def __len__(self):
        return len(self.degraded_image_list)


class TestDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(TestDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path

        all_filenames = sorted([f for f in os.listdir(degraded_path) if f.endswith(".png")],
                               key=lambda x: int(os.path.splitext(x)[0]))
        self.degraded_image_list = all_filenames[80:]  # testing on remaining 20

    def __getitem__(self, index):
        degraded_name = self.degraded_image_list[index]
        clear_name = get_matching_clear_name(degraded_name)

        degraded_img = Image.open(os.path.join(self.degraded_path, degraded_name)).convert('RGB')
        clear_img = Image.open(os.path.join(self.clear_path, clear_name)).convert('RGB')

        to_tensor = ToTensor()
        degraded_tensor = to_tensor(degraded_img)
        clear_tensor = to_tensor(clear_img)

        return degraded_tensor, clear_tensor, degraded_name

    def __len__(self):
        return len(self.degraded_image_list)


class ValDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(ValDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path

        self.degraded_image_list = sorted([f for f in os.listdir(degraded_path) if f.endswith(".png")],
                                          key=lambda x: int(os.path.splitext(x)[0]))

    def __getitem__(self, index):
        degraded_name = self.degraded_image_list[index]
        clear_name = get_matching_clear_name(degraded_name)

        degraded_img = Image.open(os.path.join(self.degraded_path, degraded_name)).convert('RGB')
        clear_img = Image.open(os.path.join(self.clear_path, clear_name)).convert('RGB')

        to_tensor = ToTensor()
        degraded_tensor = to_tensor(degraded_img)
        clear_tensor = to_tensor(clear_img)

        return {
            'degraded': degraded_tensor,
            'clear': clear_tensor,
            'filename': degraded_name
        }

    def __len__(self):
        return len(self.degraded_image_list)


/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


# **Data-loader for h-haze**

In [16]:
import os
import random
import numpy as np
import torch.utils.data as data
from PIL import Image
from torchvision.transforms import ToTensor
from albumentations import RandomCrop


def get_matching_clear_name(degraded_name):
    if "_hazy" in degraded_name:
        return degraded_name.replace("_hazy", "_GT")
    return None


class TrainDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(TrainDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path

        all_filenames = sorted([f for f in os.listdir(degraded_path) if f.endswith(".png")])
        self.degraded_image_list = all_filenames[:45]  # training on first 45

    def __getitem__(self, index):
        degraded_name = self.degraded_image_list[index]
        clear_name = get_matching_clear_name(degraded_name)

        degraded_img = Image.open(os.path.join(self.degraded_path, degraded_name)).convert('RGB')
        clear_img = Image.open(os.path.join(self.clear_path, clear_name)).convert('RGB')

        degraded_np = np.array(degraded_img)
        clear_np = np.array(clear_img)

        # Random crop using Albumentations
        crop = RandomCrop(height=256, width=256)
        augmented = crop(image=degraded_np, mask=clear_np)
        degraded_np = augmented['image']
        clear_np = augmented['mask']

        degraded_img = Image.fromarray(degraded_np)
        clear_img = Image.fromarray(clear_np)

        # Random 0°, 90°, 180°, 270° rotation
        angle = random.randint(0, 3) * 90
        degraded_img = degraded_img.rotate(angle)
        clear_img = clear_img.rotate(angle)

        to_tensor = ToTensor()
        degraded_tensor = to_tensor(degraded_img)
        clear_tensor = to_tensor(clear_img)

        return degraded_tensor, clear_tensor

    def __len__(self):
        return len(self.degraded_image_list)


class TestDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(TestDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path

        all_filenames = sorted([f for f in os.listdir(degraded_path) if f.endswith(".png")])
        self.degraded_image_list = all_filenames[45:]  # testing on remaining 10

    def __getitem__(self, index):
        degraded_name = self.degraded_image_list[index]
        clear_name = get_matching_clear_name(degraded_name)

        degraded_img = Image.open(os.path.join(self.degraded_path, degraded_name)).convert('RGB')
        clear_img = Image.open(os.path.join(self.clear_path, clear_name)).convert('RGB')

        to_tensor = ToTensor()
        degraded_tensor = to_tensor(degraded_img)
        clear_tensor = to_tensor(clear_img)

        return degraded_tensor, clear_tensor ,  degraded_name

    def __len__(self):
        return len(self.degraded_image_list)


class ValDataset(data.Dataset):
    def __init__(self, degraded_path, clear_path):
        super(ValDataset, self).__init__()
        self.degraded_path = degraded_path
        self.clear_path = clear_path

        self.degraded_image_list = sorted([f for f in os.listdir(degraded_path) if f.endswith(".png")])

    def __getitem__(self, index):
        degraded_name = self.degraded_image_list[index]
        clear_name = get_matching_clear_name(degraded_name)

        degraded_img = Image.open(os.path.join(self.degraded_path, degraded_name)).convert('RGB')
        clear_img = Image.open(os.path.join(self.clear_path, clear_name)).convert('RGB')

        to_tensor = ToTensor()
        degraded_tensor = to_tensor(degraded_img)
        clear_tensor = to_tensor(clear_img)

        return {
            'degraded': degraded_tensor,
            'clear': clear_tensor,
            'filename': degraded_name
        }

    def __len__(self):
        return len(self.degraded_image_list)


# **PROMPTblock**

In [22]:
!pip install timm
!pip install transformers

In [23]:
# !pip install timm==0.9.12 albumentations==2.0.4

In [24]:
!pip install torchvision
!pip install timm
!pip install transformers
!pip install albumentations==2.0.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 7.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.7/632.7 kB 23.9 MB/s eta 0:00:00
  Attempting uninstall: albucore
    Found existing installation: albucore 0.0.19
    Uninstalling albucore-0.0.19:
      Successfully uninstalled albucore-0.0.19
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.20
    Uninstalling albumentations-1.4.20:
      Successfully uninstalled albumentations-1.4.20


In [25]:
import os
import cv2
import gc
import numpy as np
import pandas as pd
import itertools
from tqdm.autonotebook import tqdm
import albumentations as A
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
import timm
from transformers import DistilBertModel, DistilBertConfig, DistilBertTokenizer

In [26]:
import os
import shutil

# Define the source folders
source_base = "/kaggle/input/promir-train/data/Train/Dehaze/synthetic"
parts = ["part1", "part2", "part3", "part4"]

import os
import shutil
import random

# Define the destination folder
destination_folder = "/kaggle/working/synthetic"
os.makedirs(destination_folder, exist_ok=True)  # Create the folder if it doesn't exist

# Set random seed for reproducibility (optional)
random.seed(42)

# Iterate through each part and randomly select 2500 images
for part in parts:
    source_folder = os.path.join(source_base, part)
    images = [f for f in os.listdir(source_folder) if f.endswith(('.jpg', '.png', '.bmp'))]
    
    if len(images) < 2000:
        raise ValueError(f"Not enough images in {source_folder}. Found {len(images)}, but need 2000.")
    
    # Randomly select 2500 images
    selected_images = random.sample(images, 2000)
    
    for image in selected_images:
        src_path = os.path.join(source_folder, image)
        dest_path = os.path.join(destination_folder, image)
        shutil.copy(src_path, dest_path)

print(f"Copied random 2000 images from {', '.join(parts)} to {destination_folder}")



Copied random 2000 images from part1, part2, part3, part4 to /kaggle/working/synthetic


In [27]:
import os

# Specify the folder path
folder_path =  "/kaggle/working/synthetic" # Replace with your folder path

# List all files in the folder
files = os.listdir(folder_path)

# Filter to include only image files (optional)
image_files = [f for f in files if f.endswith(('.jpg', '.png', '.bmp'))]

# Get the length
length = len(image_files)

print(f"The folder '{folder_path}' contains {length} images.")


The folder '/kaggle/working/synthetic' contains 8000 images.


In [28]:
import os
import cv2

folder_path =  "/kaggle/working/synthetic"
def check_invalid_images(folder_path):
    invalid_images = []  # List to store paths of invalid images
    valid_extensions = ('.jpg', '.png', '.bmp', '.jpeg')  # Add supported extensions if needed

    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            
            # Check if the file has a valid image extension
            if not file.lower().endswith(valid_extensions):
                print(f"Skipping non-image file: {file_path}")
                continue
            
            # Attempt to read the image
            image = cv2.imread(file_path)
            if image is None:
                print(f"Invalid image found: {file_path}")
                invalid_images.append(file_path)
    
    if invalid_images:
        print(f"\nTotal invalid images: {len(invalid_images)}")
    else:
        print("All images are valid!")
    
    return invalid_images

invalid_images = check_invalid_images(folder_path)


All images are valid!


In [29]:
import os
import shutil

# Paths
input_path = "/kaggle/working/"
output_images_path = "/kaggle/working/8kk/images"
captions_file_path = "/kaggle/working/8kk/captions.txt"

# Create output directories
os.makedirs(output_images_path, exist_ok=True)

# Labels mapping based on folder
folder_labels = {
    "synthetic": "hazy",

}

# Initialize captions.txt
with open(captions_file_path, "w") as captions_file:
    captions_file.write("image,caption\n")  # Header for the captions file

    # Iterate through the folder structure
    for folder_name, label in folder_labels.items():
        folder_path = os.path.join(input_path, folder_name)
        
        if folder_name == "synthetic": 
            for file_name in os.listdir(folder_path):
                src = os.path.join(folder_path, file_name)
                dest = os.path.join(output_images_path, file_name)
                shutil.copy(src, dest)
                captions_file.write(f"{file_name},{label}\n")# Handle synthetic folder separately
            
        else:
            pass

print("Dataset successfully formatted!")


import os

# Specify the folder path
folder_path =  "/kaggle/working/8kk/images" # Replace with your folder path

# List all files in the folder
files = os.listdir(folder_path)

# Filter to include only image files (optional)
image_files = [f for f in files if f.endswith(('.jpg', '.png', '.bmp'))]

# Get the length
length = len(image_files)

print(f"The folder '{folder_path}' contains {length} images.")

Dataset successfully formatted!
The folder '/kaggle/working/8kk/images' contains 8000 images.


In [30]:
import os
import cv2
from albumentations import (
    HorizontalFlip, Rotate, RandomBrightnessContrast, Blur, GaussNoise, Compose, RandomCrop
)
from albumentations.core.composition import OneOf

# Paths
rainy_folder = "/kaggle/input/promir-train/data/Train/Derain/rainy"
augmented_folder_new = "/kaggle/working/augmented_new/rainy_new"

# Create the output folder for augmented images
os.makedirs(augmented_folder_new, exist_ok=True)

# Define augmentations using Albumentations
augmentations = Compose([
    HorizontalFlip(p=0.5),
    Rotate(limit=30, p=0.7),  # Rotate images randomly within ±30 degrees
    RandomCrop(height=256, width=256, p=0.5)
])

# Count images
original_images = [file for file in os.listdir(rainy_folder) if file.endswith(('.jpg', '.png', '.bmp'))]
num_original = len(original_images)
required_images = 2000
augmentations_per_image = (required_images // num_original) + 1  # Calculate how many augmented images to create

print(f"Original images: {num_original}, Augmented per image: {augmentations_per_image}")

# Loop through rainy folder images
for file_name in original_images:
    file_path = os.path.join(rainy_folder, file_name)
    image = cv2.imread(file_path)
    if image is not None:
        # Apply augmentations
        for i in range(augmentations_per_image):
            augmented = augmentations(image=image)["image"]
            # Save augmented image
            augmented_file_name = f"{os.path.splitext(file_name)[0]}_aug{i}.jpg"
            augmented_file_path = os.path.join(augmented_folder_new, augmented_file_name)
            cv2.imwrite(augmented_file_path, augmented)

print("Augmentation complete! Augmented images saved to:", augmented_folder_new)



Original images: 200, Augmented per image: 11
Augmentation complete! Augmented images saved to: /kaggle/working/augmented_new/rainy_new


In [31]:
import os

# Specify the folder path
folder_path =  "/kaggle/working/augmented_new/rainy_new" # Replace with your folder path

# List all files in the folder
files = os.listdir(folder_path)

# Filter to include only image files (optional)
image_files = [f for f in files if f.endswith(('.jpg', '.png', '.bmp'))]

# Get the length
length = len(image_files)

print(f"The folder '{folder_path}' contains {length} images.")

The folder '/kaggle/working/augmented_new/rainy_new' contains 2200 images.


In [32]:
input_path1 = "/kaggle/working/augmented_new"



folder_labels = {"rainy_new": "rainy"
                }


with open(captions_file_path, "a") as captions_file:


    # Iterate through the folder structure
    for folder_name, label in folder_labels.items():
        folder_path = os.path.join(input_path1, folder_name)
        
        if folder_name == "rainy_new": 
            for file_name in os.listdir(folder_path):
                src = os.path.join(folder_path, file_name)
                dest = os.path.join(output_images_path, file_name)
                shutil.copy(src, dest)
                captions_file.write(f"{file_name},{label}\n")# Handle synthetic folder separately
            
        else:
            pass

print("Dataset successfully formatted!")
# Handle rainy and Denoise folders
    

Dataset successfully formatted!


In [33]:
import os

# Specify the folder path
folder_path =  "/kaggle/working/8kk/images" # Replace with your folder path

# List all files in the folder
files = os.listdir(folder_path)

# Filter to include only image files (optional)
image_files = [f for f in files if f.endswith(('.jpg', '.png', '.bmp'))]

# Get the length
length = len(image_files)

print(f"The folder '{folder_path}' contains {length} images.")

The folder '/kaggle/working/8kk/images' contains 10200 images.


In [34]:

input_path = "/kaggle/input/promir-train/data/Train/"

folder_labels = {    "Denoise": "noisy "
                }


with open(captions_file_path, "a") as captions_file:


    # Iterate through the folder structure
    for folder_name, label in folder_labels.items():
        folder_path = os.path.join(input_path, folder_name)
        
        if folder_name == "Denoise": 
            for file_name in os.listdir(folder_path):
                src = os.path.join(folder_path, file_name)
                dest = os.path.join(output_images_path, file_name)
                shutil.copy(src, dest)
                captions_file.write(f"{file_name},{label}\n")# Handle synthetic folder separately
            
        else:
            pass

print("Dataset successfully formatted!")
# Handle rainy and Denoise folders

Dataset successfully formatted!


In [35]:
import os

# Specify the folder path
folder_path =  "/kaggle/working/8kk/images" # Replace with your folder path

# List all files in the folder
files = os.listdir(folder_path)

# Filter to include only image files (optional)
image_files = [f for f in files if f.endswith(('.jpg', '.png', '.bmp'))]

# Get the length
length = len(image_files)

print(f"The folder '{folder_path}' contains {length} images.")

The folder '/kaggle/working/8kk/images' contains 14922 images.


In [36]:
import pandas as pd

In [37]:

df = pd.read_csv("/kaggle/working/8kk/captions.txt")

    # Adjust the list comprehension to match the number of rows
df['id'] = [id_ for id_ in range(df.shape[0] // 5 + 1) for _ in range(5)][:df.shape[0]]
df.to_csv("captions.csv", index=False)
df = pd.read_csv("captions.csv")
image_path = "/kaggle/working/8kk/images"
captions_path = "/kaggle/working"
df.head()

,image,caption,id
0,5451_0.8_0.04.jpg,hazy,0
1,4419_0.95_0.12.jpg,hazy,0
2,8856_0.95_0.12.jpg,hazy,0
3,4152_0.85_0.04.jpg,hazy,0
4,8307_0.8_0.1.jpg,hazy,0


In [38]:
class CFG:
    debug = False
    image_path = image_path
    captions_path = captions_path
    batch_size = 32
    num_workers = 2
    head_lr = 1e-3
    image_encoder_lr = 1e-4
    text_encoder_lr = 1e-5
    weight_decay = 1e-3
    patience = 1
    factor = 0.8
    epochs = 30
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_name = 'resnet50'
    image_embedding = 2048
    text_encoder_model = "distilbert-base-uncased"
    text_embedding = 768
    text_tokenizer = "distilbert-base-uncased"
    max_length = 200

    pretrained = True # for both image encoder and text encoder
    trainable = True # for both image encoder and text encoder
    temperature = 1.0

    # image size
    size = 224

    # for projection head; used for both image and text encoders
    num_projection_layers = 1
    projection_dim = 256
    dropout = 0.1

In [39]:
class AvgMeter:
    def __init__(self, name="Metric"):
        self.name = name
        self.reset()

    def reset(self):
        self.avg, self.sum, self.count = [0] * 3

    def update(self, val, count=1):
        self.count += count
        self.sum += val * count
        self.avg = self.sum / self.count

    def __repr__(self):
        text = f"{self.name}: {self.avg:.4f}"
        return text

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group["lr"]


In [40]:
class CLIPDataset(torch.utils.data.Dataset):
    def __init__(self, image_filenames, captions, tokenizer, transforms):
        """
        image_filenames and cpations must have the same length; so, if there are
        multiple captions for each image, the image_filenames must have repetitive
        file names
        """

        self.image_filenames = image_filenames
        self.captions = list(captions)
        self.encoded_captions = tokenizer(
            list(captions), padding=True, truncation=True, max_length=CFG.max_length
        )
        self.transforms = transforms

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(values[idx])
            for key, values in self.encoded_captions.items()
        }

        image = cv2.imread(f"{CFG.image_path}/{self.image_filenames[idx]}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transforms(image=image)['image']
        item['image'] = torch.tensor(image).permute(2, 0, 1).float()
        item['caption'] = self.captions[idx]

        return item


    def __len__(self):
        return len(self.captions)



def get_transforms(mode="train"):
    if mode == "train":
        return A.Compose(
            [
                A.Resize(CFG.size, CFG.size, always_apply=True),
                A.Normalize(max_pixel_value=255.0, always_apply=True),
            ]
        )
    else:
        return A.Compose(
            [
                A.Resize(CFG.size, CFG.size, always_apply=True),
                A.Normalize(max_pixel_value=255.0, always_apply=True),
            ]
        )

In [41]:
class ImageEncoder(nn.Module):
    """
    Encode images to a fixed size vector
    """

    def __init__(
        self, model_name=CFG.model_name, pretrained=CFG.pretrained, trainable=CFG.trainable
    ):
        super().__init__()
        self.model = timm.create_model(
            model_name, pretrained, num_classes=0, global_pool="avg"
        )
        for p in self.model.parameters():
            p.requires_grad = trainable

    def forward(self, x):
        return self.model(x)

In [42]:
class TextEncoder(nn.Module):
    def __init__(self, model_name=CFG.text_encoder_model, pretrained=CFG.pretrained, trainable=CFG.trainable):
        super().__init__()
        if pretrained:
            self.model = DistilBertModel.from_pretrained(model_name)
        else:
            self.model = DistilBertModel(config=DistilBertConfig())

        for p in self.model.parameters():
            p.requires_grad = trainable

        # we are using the CLS token hidden representation as the sentence's embedding
        self.target_token_idx = 0

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = output.last_hidden_state
        return last_hidden_state[:, self.target_token_idx, :]

In [43]:
class ProjectionHead(nn.Module):
    def __init__(
        self,
        embedding_dim,
        projection_dim=CFG.projection_dim,
        dropout=CFG.dropout
    ):
        super().__init__()
        self.projection = nn.Linear(embedding_dim, projection_dim)
        self.gelu = nn.GELU()
        self.fc = nn.Linear(projection_dim, projection_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(projection_dim)

    def forward(self, x):
        projected = self.projection(x)
        x = self.gelu(projected)
        x = self.fc(x)
        x = self.dropout(x)
        x = x + projected
        x = self.layer_norm(x)
        return x

In [44]:
class CLIPModel(nn.Module):
    def __init__(
        self,
        temperature=CFG.temperature,
        image_embedding=CFG.image_embedding,
        text_embedding=CFG.text_embedding,
    ):
        super().__init__()
        self.image_encoder = ImageEncoder()
        self.text_encoder = TextEncoder()
        self.image_projection = ProjectionHead(embedding_dim=image_embedding)
        self.text_projection = ProjectionHead(embedding_dim=text_embedding)
        self.temperature = temperature

    def forward(self, batch):
        # Getting Image and Text Features
        image_features = self.image_encoder(batch["image"])
        text_features = self.text_encoder(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
        )
        # Getting Image and Text Embeddings (with same dimension)
        image_embeddings = self.image_projection(image_features)
        text_embeddings = self.text_projection(text_features)

        # Calculating the Loss
        logits = (text_embeddings @ image_embeddings.T) / self.temperature
        images_similarity = image_embeddings @ image_embeddings.T
        texts_similarity = text_embeddings @ text_embeddings.T
        targets = F.softmax(
            (images_similarity + texts_similarity) / 2 * self.temperature, dim=-1
        )
        texts_loss = cross_entropy(logits, targets, reduction='none')
        images_loss = cross_entropy(logits.T, targets.T, reduction='none')
        loss =  (images_loss + texts_loss) / 2.0 # shape: (batch_size)
        return loss.mean()


def cross_entropy(preds, targets, reduction='none'):
    log_softmax = nn.LogSoftmax(dim=-1)
    loss = (-targets * log_softmax(preds)).sum(1)
    if reduction == "none":
        return loss
    elif reduction == "mean":
        return loss.mean()

In [45]:
def make_train_valid_dfs():
    dataframe = pd.read_csv(f"{CFG.captions_path}/captions.csv")
    max_id = dataframe["id"].max() + 1 if not CFG.debug else 100
    image_ids = np.arange(0, max_id)
    np.random.seed(42)
    valid_ids = np.random.choice(
        image_ids, size=int(0.2 * len(image_ids)), replace=False
    )
    train_ids = [id_ for id_ in image_ids if id_ not in valid_ids]
    train_dataframe = dataframe[dataframe["id"].isin(train_ids)].reset_index(drop=True)
    valid_dataframe = dataframe[dataframe["id"].isin(valid_ids)].reset_index(drop=True)
    return train_dataframe, valid_dataframe

# train_df, valid_df = make_train_valid_dfs()
def build_loaders(dataframe, tokenizer, mode):
    transforms = get_transforms(mode=mode)
    dataset = CLIPDataset(
        dataframe["image"].values,
        dataframe["caption"].values,
        tokenizer=tokenizer,
        transforms=transforms,
    )
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
        shuffle=True if mode == "train" else False,
    )
    return dataloader

In [46]:
def get_image_embeddings(valid_df, model_path):
    tokenizer = DistilBertTokenizer.from_pretrained(CFG.text_tokenizer)
    valid_loader = build_loaders(valid_df, tokenizer, mode="valid")

    model = CLIPModel().to(CFG.device)
    model.load_state_dict(torch.load(model_path, map_location=CFG.device))
    model.eval()

    valid_image_embeddings = []
    with torch.no_grad():
        for batch in tqdm(valid_loader):
            image_features = model.image_encoder(batch["image"].to(CFG.device))
            image_embeddings = model.image_projection(image_features)
            valid_image_embeddings.append(image_embeddings)
    return model, torch.cat(valid_image_embeddings)

In [47]:
_, valid_df = make_train_valid_dfs()
model, image_embeddings = get_image_embeddings(valid_df, "/kaggle/input/prompt_trained_model/pytorch/default/1/best (1).pt")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

<ipython-input-46-95b5c045d3a7>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=CFG.device))


  0%|          | 0/94 [00:00<?, ?it/s]

In [48]:
# def find_matches(model, image_embeddings, image_filenames, query_image_path, n=9):
#     """
#     Finds the closest text captions to a given query image.

#     Args:
#         model: The trained CLIP model.
#         image_embeddings: Pre-computed image embeddings for the dataset.
#         image_filenames: A list of image filenames corresponding to the embeddings.
#         query_image_path: The path to the query image.
#         n: The number of closest captions to retrieve.

#     Returns:
#         A list of the `n` closest captions to the query image.
#     """
#     # 1. Encode the query image:
#     query_image = cv2.imread(query_image_path)
#     query_image = cv2.cvtColor(query_image, cv2.COLOR_BGR2RGB)
#     query_image = get_transforms(mode="valid")(image=query_image)['image']
#     query_image = torch.tensor(query_image).permute(2, 0, 1).float().unsqueeze(0).to(CFG.device)  # Add batch dimension

#     with torch.no_grad():
#         query_image_features = model.image_encoder(query_image)
#         query_image_embedding = model.image_projection(query_image_features)

#     # 2. Calculate similarities:
#     image_embeddings_n = F.normalize(image_embeddings, p=2, dim=-1)
#     query_image_embedding_n = F.normalize(query_image_embedding, p=2, dim=-1)
#     dot_similarity = query_image_embedding_n @ image_embeddings_n.T  

#     # 3. Get top matches:
#     values, indices = torch.topk(dot_similarity.squeeze(0), n)
#     matches = [valid_df['caption'].values[idx] for idx in indices]  # Retrieve captions instead of filenames

#     return matches

In [49]:
import os
import cv2
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from torchvision.transforms import ToTensor

class ImageCaptionMatcher:
    def __init__(self, model, image_embeddings, valid_df, device="cuda" if torch.cuda.is_available() else "cpu"):
        """
        Initialize the ImageCaptionMatcher class.

        Args:
            model: The trained CLIP model.
            image_embeddings: Precomputed image embeddings for the dataset.
            valid_df: DataFrame containing image filenames and captions.
            device: The device to run computations on (default: "cuda" if available, else "cpu").
        """
        self.model = model.to(device)
        self.image_embeddings = F.normalize(image_embeddings, p=2, dim=-1)
        self.valid_df = valid_df
        self.device = device
    
    def preprocess_image(self, image_path):
        """Loads and preprocesses the input image."""
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Error: Unable to load the image at {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = get_transforms(mode="valid")(image=image)['image']
        image = torch.tensor(image).permute(2, 0, 1).float().unsqueeze(0).to(self.device)
        return image
    
    def find_closest_caption(self, query_image_paths, n=1):
        """
        Finds the closest captions for a tuple of query image paths.

        Args:
            query_image_paths: Tuple of image paths.
            n: Number of closest captions to retrieve per image (default: 1).

        Returns:
            A tuple of the `n` closest captions for each query image.
        """
        closest_captions_list = []
        
        for query_image_path in query_image_paths:
            # Encode the query image
            query_image = self.preprocess_image(query_image_path)
            
            with torch.no_grad():
                query_image_features = self.model.image_encoder(query_image)
                query_image_embedding = F.normalize(self.model.image_projection(query_image_features), p=2, dim=-1)
            
            # Compute similarity scores
            similarity_scores = query_image_embedding @ self.image_embeddings.T
            values, indices = torch.topk(similarity_scores.squeeze(0), n)
            
            # Retrieve the closest captions
            closest_captions = [self.valid_df['caption'].values[idx] for idx in indices]
            closest_captions_list.append(closest_captions)
            # print(closest_captions)
        
        return tuple(closest_captions_list)


In [50]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import CLIPModel, AutoTokenizer




# class PromptGenBlock(nn.Module):
#     def __init__(self, prompt_dim, prompt_len=512, prompt_size=32, lin_dim=16, device="cuda"):
#         super(PromptGenBlock, self).__init__()
#         self.prompt_dim = prompt_dim
#         self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        
#         # Learnable prompt parameter
#         self.prompt_param = nn.Parameter(torch.rand(1, prompt_len, prompt_dim, prompt_size, prompt_size))
        
#         # Linear layer for text feature mapping
#         self.linear_layer = nn.Linear(lin_dim, prompt_len)
        
#         # Convolution layer to refine prompt features
#         self.conv3x3 = nn.Conv2d(prompt_dim, prompt_dim, kernel_size=3, stride=1, padding=1, bias=False)
        
#         # Load CLIP model and tokenizer
#         self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
#         self.model = self.model.to(self.device)
#         self.tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

#     def generate_text_features(self, text_prompt):
#         """Generate text features from a given text prompt."""
#         inputs = self.tokenizer([text_prompt], padding=True, return_tensors="pt")
#         inputs = {k: v.to(self.device) for k, v in inputs.items()}
#         text_features = self.model.get_text_features(**inputs)
#         return text_features.to(self.device)

#     def forward(self, x, closest_captions_tuple):
#         B, C, H, W = x.shape  # Expected: (B, 64, H/2, W/2)
#         prompt_dim = 2 * C
        
#         # Initialize accumulated prompt tensor
#         accumulated_prompt = None
        
#         for closest_captions in closest_captions_tuple:
#             # Generate text features
#             text_prompt = f"a {closest_captions[0]} photo of a *"
#             text_features = self.generate_text_features(text_prompt)
            
#             # Adjust text feature dimensions
#             prompt_weights = text_features[0].unsqueeze(-2)  # Shape: (token_count, hidden_dim, 1)
            
#             # Generate weighted prompts
#             prompt = (prompt_weights.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1) * 
#                      self.prompt_param.unsqueeze(0).repeat(B, 1, 1, 1, 1, 1).squeeze(1))
            
#             # Sum over prompt_len dimension
#             prompt = torch.sum(prompt, dim=1)  
            
#             # Downsample to (H/4, W/4)
#             prompt = F.interpolate(prompt, (int(H), int(W)), mode="bilinear")
            
#             # Apply convolution
#             prompt = self.conv3x3(prompt)
            
#             # Accumulate prompts
#             if accumulated_prompt is None:
#                 accumulated_prompt = prompt
#             else:
#                 accumulated_prompt = accumulated_prompt + prompt
        
#         # Average the accumulated prompts if there were multiple captions
#         if len(closest_captions_tuple) > 1:
#             accumulated_prompt = accumulated_prompt / len(closest_captions_tuple)
#         print(accumulated_prompt.shape)
#         return accumulated_prompt

In [51]:
from transformers import CLIPModel, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

def get_text_embedding(text):
    inputs = tokenizer([text], padding=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        text_emb = clip_model.get_text_features(**inputs)
    return text_emb  # shape: (1, hidden_dim)

f1= get_text_embedding("a hazy photo"),
f2= get_text_embedding("a noisy photo"),
f3= get_text_embedding("a rainy photo")

text_embeddings = {
    "hazy": f1,
    'noisy': f2,
    "rainy": f3
}
# print(f1)

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [52]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import CLIPModel, AutoTokenizer

# class PromptGenBlock(nn.Module):
#     def __init__(self, prompt_dim, prompt_len=128, prompt_size=32, lin_dim=16, device="cuda"):
#         super(PromptGenBlock, self).__init__()
#         self.prompt_dim = prompt_dim
#         self.device = torch.device(device if torch.cuda.is_available() else "cpu")

#         # Learnable prompt parameter
#         self.prompt_param = nn.Parameter(torch.rand(1, prompt_len, prompt_dim, prompt_size, prompt_size))

#         # Linear layer for text feature mapping
#         self.linear_layer = nn.Linear(lin_dim, prompt_len)

#         # Convolution layer to refine prompt features
#         self.conv3x3 = nn.Conv2d(prompt_dim, prompt_dim, kernel_size=3, stride=1, padding=1, bias=False)

#     text_embeddings = {
#         "hazy": f1,
#         "noisy": f2,
#         "rainy": f3
#     }

#     def forward(self, x, closest_captions_tuple):
#         B, C, H, W = x.shape  # Expected: (B, 64, H/2, W/2)
#         prompts_list = []  # Store individual prompts in a tensor-compatible format

#         for closest_captions in closest_captions_tuple:
#             text_prompt = closest_captions[0].strip()  # Remove extra spaces

#             # Get text features from precomputed embeddings
#             text_features = self.text_embeddings.get(text_prompt)[0][0]
#             if text_features is None:
#                 raise ValueError(f"Text prompt '{text_prompt}' not found in text_embeddings dictionary")

#             # Ensure text_features is (hidden_dim,)
#             # text_features = torch.stack(text_features)
#             text_features = text_features.squeeze()  #

#             # Ensure prompt_weights has shape (hidden_dim, 1, 1)
#             prompt_weights = text_features[..., None, None]  # Adds two singleton dimensions
            
#             # Generate weighted prompt (adjusted to correct dimensions)
#             prompt = (prompt_weights * self.prompt_param.squeeze(0))  # (hidden_dim, H, W)

#             # Sum over prompt_len dimension
#             prompt = torch.sum(prompt, dim=0)  # Shape: (prompt_dim, H, W)

#             # Expand to match batch size
#             prompt = prompt.unsqueeze(0).expand(B, -1, -1, -1)  # Shape: (B, prompt_dim, H, W)

#             # Downsample to (H, W)
#             prompt = F.interpolate(prompt, size=(int(H/2), int(W/2)), mode="bilinear", align_corners=False)

#             # Apply convolution (expects 4D input)
#             prompt = self.conv3x3(prompt)

#             # Store the prompt in a tensor list
#             prompts_list.append(prompt)

#         # # Stack prompts into a single tensor
#         prompts_tensor = torch.stack(prompts_list, dim=1)  # Shape: (B, num_captions, prompt_dim, H, W)
#         print(prompts_tensor.shape)
#         return prompts_tensor


In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PromptGenBlock(nn.Module):
    def __init__(self, prompt_dim, prompt_len=128, prompt_size=32, lin_dim=16, device="cuda"):
        super(PromptGenBlock, self).__init__()
        self.prompt_dim = prompt_dim
        self.prompt_len = prompt_len  # Ensure it matches text embeddings' dim
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        # Learnable prompt parameter (matches text embedding dim)
        self.prompt_param = nn.Parameter(torch.rand(1, prompt_len, prompt_dim, prompt_size, prompt_size))

        # Linear layer for text feature mapping (512 → prompt_len)
        self.linear_layer = nn.Linear(512, prompt_len)  # Match text embedding size

        # Convolution layer to refine prompt features
        self.conv3x3 = nn.Conv2d(prompt_dim, prompt_dim, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv1x1 = nn.Conv2d(512, 128, kernel_size=1)

    text_embeddings = {
        "hazy": f1,
        "noisy": f2,
        "rainy": f3
    }

    def forward(self, x, closest_captions_tuple):
        B, C, H, W = x.shape  # Expected: (B, 64, H/2, W/2)
        
        batch_text_features = []
        for i in range(B):
            text_prompt = closest_captions_tuple[i][0].strip()  

            # Get text features (512-dim)
            text_features = self.text_embeddings.get(text_prompt)
            if text_features is None:
                raise ValueError(f"Text prompt '{text_prompt}' not found in text_embeddings dictionary")

            text_features = text_features[0][0].squeeze()  # Shape: (512,)
            batch_text_features.append(text_features)

        # Convert batch_text_features to a tensor
        batch_text_features = torch.stack(batch_text_features).to(self.device)  # Shape: (B, 512)

        # Map text features to prompt_len (128 or 512)
        batch_text_features = self.linear_layer(batch_text_features)  # Shape: (B, prompt_len)

        # Expand to match prompt_param shape
        prompt_weights = batch_text_features[..., None, None, None]  # Shape: (B, prompt_len, 1, 1, 1)

        # Generate weighted prompts
        prompt = (prompt_weights * self.prompt_param)  # Shape: (B, prompt_len, prompt_dim, H, W)

        # Sum over prompt_len
        prompt = torch.sum(prompt, dim=1)  # Shape: (B, prompt_dim, H, W)

        # Downsample to (H/2, W/2)
        prompt = F.interpolate(prompt, size=(H, W), mode="bilinear", align_corners=False)

        # Apply convolution
        prompt = self.conv3x3(prompt)
        prompt= self.conv1x1(prompt)
        # print(prompt.shape)
        return prompt  # Shape: (B, prompt_dim, H/2, W/2)


In [54]:
# from transformers import AutoTokenizer, CLIPModel

# model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

# inputs = tokenizer(["a dehazed photo of a *", "a derained photo of a *", "a denoised photo of a *"], padding=True, return_tensors="pt")
# text_features = model.get_text_features(**inputs)



# #########################################################################

# import torch
# from torch import nn
# import torch.nn.functional as F
# Batch = 1
# class PromptGenBlock(nn.Module):
#     def __init__(self,prompt_dim=128,prompt_len=512,prompt_size = 16,lin_dim = 16, H= 64, W=64, Batch = 2):
#         super(PromptGenBlock,self).__init__()
#         self.H = H
#         self.W = W
#     #     self.prompt_param = nn.Parameter(torch.rand(1,prompt_len,prompt_dim,prompt_size,prompt_size))
#     #     print('prompt param:', self.prompt_param.shape)
#         self.linear_layer = nn.Linear(lin_dim,prompt_len)
#     #     self.conv3x3 = nn.Conv2d(prompt_dim,prompt_dim,kernel_size=3,stride=1,padding=1,bias=False)
        

#     def forward(self,x):
#         B,C,H,W = x.shape
#         emb = x.mean(dim=(-2,-1))
#         prompt_weights = F.softmax(self.linear_layer(emb),dim=1)
#         prompt = prompt_weights * text_features[0]
#         # prompt = torch.sum(prompt,dim=1)
#         print("prompt shape:", prompt.shape)
#         prompt = F.interpolate(prompt,(self.H,self.W),mode="bilinear")
#         prompt = self.conv3x3(prompt)
#         return prompt


# #######################################################################
# net = PromptGenBlock(prompt_dim=128, lin_dim = 16, H=64, W=64)
# x = torch.randn((1,16,64,64))
# # y = torch.randn((1,512,64,64))
# out = net(x)
# print('Output size:', out.shape)

# **Original arch**

In [12]:
import torch.nn as nn
import torch.nn.functional as F

# from .modules import DEABlockTrain, DEBlockTrain, CGAFusion


def default_conv(in_channels, out_channels, kernel_size, bias=True):
    return nn.Conv2d(in_channels, out_channels, kernel_size, padding=(kernel_size // 2), bias=bias)


class DEANet(nn.Module):
    def __init__(self, base_dim=32):
        super(DEANet, self).__init__()
        # down-sample
        self.down1 = nn.Sequential(nn.Conv2d(3, base_dim, kernel_size=3, stride = 1, padding=1))
        self.down2 = nn.Sequential(nn.Conv2d(base_dim, base_dim*2, kernel_size=3, stride=2, padding=1),
                                   nn.ReLU(True))
        self.down3 = nn.Sequential(nn.Conv2d(base_dim*2, base_dim*4, kernel_size=3, stride=2, padding=1),
                                   nn.ReLU(True))
        # level1
        self.down_level1_block1 = DEBlockTrain(default_conv, base_dim, 3)
        self.down_level1_block2 = DEBlockTrain(default_conv, base_dim, 3)
        self.down_level1_block3 = DEBlockTrain(default_conv, base_dim, 3)
        self.down_level1_block4 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block1 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block2 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block3 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block4 = DEBlockTrain(default_conv, base_dim, 3)
        # level2
        self.fe_level_2 = nn.Conv2d(in_channels=base_dim * 2, out_channels=base_dim * 2, kernel_size=3, stride=1, padding=1)
        self.down_level2_block1 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.down_level2_block2 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.down_level2_block3 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.down_level2_block4 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block1 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block2 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block3 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block4 = DEBlockTrain(default_conv, base_dim * 2, 3)
        # level3
        self.fe_level_3 = nn.Conv2d(in_channels=base_dim * 4, out_channels=base_dim * 4, kernel_size=3, stride=1, padding=1)
        self.level3_block1 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block2 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block3 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block4 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block5 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block6 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block7 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block8 = DEABlockTrain(default_conv, base_dim * 4, 3)
        # up-sample
        self.up1 = nn.Sequential(nn.ConvTranspose2d(base_dim*4, base_dim*2, kernel_size=3, stride=2, padding=1, output_padding=1),
                                 nn.ReLU(True))
        self.up2 = nn.Sequential(nn.ConvTranspose2d(base_dim*2, base_dim, kernel_size=3, stride=2, padding=1, output_padding=1),
                                 nn.ReLU(True))
        self.up3 = nn.Sequential(nn.Conv2d(base_dim, 3, kernel_size=3, stride=1, padding=1))
        # feature fusion
        self.mix1 = CGAFusion(base_dim * 4, reduction=8)
        self.mix2 = CGAFusion(base_dim * 2, reduction=4)

    def forward(self, x):
        x_down1 = self.down1(x)
        x_down1 = self.down_level1_block1(x_down1)
        x_down1 = self.down_level1_block2(x_down1)
        x_down1 = self.down_level1_block3(x_down1)
        x_down1 = self.down_level1_block4(x_down1)

        x_down2 = self.down2(x_down1)
        x_down2_init = self.fe_level_2(x_down2)
        x_down2_init = self.down_level2_block1(x_down2_init)
        x_down2_init = self.down_level2_block2(x_down2_init)
        x_down2_init = self.down_level2_block3(x_down2_init)
        x_down2_init = self.down_level2_block4(x_down2_init)

        x_down3 = self.down3(x_down2_init)
        x_down3_init = self.fe_level_3(x_down3)
        x1 = self.level3_block1(x_down3_init)
        x2 = self.level3_block2(x1)
        x3 = self.level3_block3(x2)
        x4 = self.level3_block4(x3)
        x5 = self.level3_block5(x4)
        x6 = self.level3_block6(x5)
        x7 = self.level3_block7(x6)
        x8 = self.level3_block8(x7)
        x_level3_mix = self.mix1(x_down3, x8)

        x_up1 = self.up1(x_level3_mix)
        x_up1 = self.up_level2_block1(x_up1)
        x_up1 = self.up_level2_block2(x_up1)
        x_up1 = self.up_level2_block3(x_up1)
        x_up1 = self.up_level2_block4(x_up1)

        x_level2_mix = self.mix2(x_down2, x_up1)
        x_up2 = self.up2(x_level2_mix)
        x_up2 = self.up_level1_block1(x_up2)
        x_up2 = self.up_level1_block2(x_up2)
        x_up2 = self.up_level1_block3(x_up2)
        x_up2 = self.up_level1_block4(x_up2)
        out = self.up3(x_up2)

        return out

# **Architecture.py**

In [55]:
import torch.nn as nn
import torch.nn.functional as F

# from .modules import DEABlockTrain, DEBlockTrain, CGAFusion


def default_conv(in_channels, out_channels, kernel_size, bias=True):
    return nn.Conv2d(in_channels, out_channels, kernel_size, padding=(kernel_size // 2), bias=bias)


class DEANet(nn.Module):
    def __init__(self, base_dim=32):
        super(DEANet, self).__init__()
        # down-sample
        self.down1 = nn.Sequential(nn.Conv2d(3, base_dim, kernel_size=3, stride = 1, padding=1))
        self.down2 = nn.Sequential(nn.Conv2d(base_dim, base_dim*2, kernel_size=3, stride=2, padding=1),
                                   nn.ReLU(True))
        self.down3 = nn.Sequential(nn.Conv2d(base_dim*2, base_dim*4, kernel_size=3, stride=2, padding=1),
                                   nn.ReLU(True))
        # level1
        self.down_level1_block1 = DEBlockTrain(default_conv, base_dim, 3)
        self.down_level1_block2 = DEBlockTrain(default_conv, base_dim, 3)
        self.down_level1_block3 = DEBlockTrain(default_conv, base_dim, 3)
        self.down_level1_block4 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block1 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block2 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block3 = DEBlockTrain(default_conv, base_dim, 3)
        self.up_level1_block4 = DEBlockTrain(default_conv, base_dim, 3)
        # level2
        self.fe_level_2 = nn.Conv2d(in_channels=base_dim * 2, out_channels=base_dim * 2, kernel_size=3, stride=1, padding=1)
        self.down_level2_block1 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.down_level2_block2 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.down_level2_block3 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.down_level2_block4 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block1 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block2 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block3 = DEBlockTrain(default_conv, base_dim * 2, 3)
        self.up_level2_block4 = DEBlockTrain(default_conv, base_dim * 2, 3)
        # level3
        self.fe_level_3 = nn.Conv2d(in_channels=base_dim * 4, out_channels=base_dim * 4, kernel_size=3, stride=1, padding=1)
        self.level3_block1 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block2 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block3 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block4 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block5 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block6 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block7 = DEABlockTrain(default_conv, base_dim * 4, 3)
        self.level3_block8 = DEABlockTrain(default_conv, base_dim * 4, 3)
        # up-sample
        self.up1 = nn.Sequential(nn.ConvTranspose2d(base_dim*4, base_dim*2, kernel_size=3, stride=2, padding=1, output_padding=1),
                                 nn.ReLU(True))
        self.up2 = nn.Sequential(nn.ConvTranspose2d(base_dim*2, base_dim, kernel_size=3, stride=2, padding=1, output_padding=1),
                                 nn.ReLU(True))
        self.up3 = nn.Sequential(nn.Conv2d(base_dim, 3, kernel_size=3, stride=1, padding=1))
        # prompt generation 
        self.prompt = PromptGenBlock(prompt_dim=512, lin_dim = 16)
        # self.caption = ImageCaptionMatcher
        self.matcher = ImageCaptionMatcher(model, image_embeddings, valid_df)
        # feature fusion
        self.mix1 = CGAFusion(base_dim * 4, reduction=8)
        self.mix2 = CGAFusion(base_dim * 2, reduction=4)

    def forward(self, x,query_image_path):
        x_down1 = self.down1(x)
        x_down1 = self.down_level1_block1(x_down1)
        x_down1 = self.down_level1_block2(x_down1)
        x_down1 = self.down_level1_block3(x_down1)
        x_down1 = self.down_level1_block4(x_down1)

        x_down2 = self.down2(x_down1)
        x_down2_init = self.fe_level_2(x_down2)
        x_down2_init = self.down_level2_block1(x_down2_init)
        x_down2_init = self.down_level2_block2(x_down2_init)
        x_down2_init = self.down_level2_block3(x_down2_init)
        x_down2_init = self.down_level2_block4(x_down2_init)

        x_down3 = self.down3(x_down2_init)
        x_down3_init = self.fe_level_3(x_down3)
        x1 = self.level3_block1(x_down3_init)
        x2 = self.level3_block2(x1)
        x3 = self.level3_block3(x2)
        x4 = self.level3_block4(x3)
        x5 = self.level3_block5(x4)
        x6 = self.level3_block6(x5)
        x7 = self.level3_block7(x6)
        x8 = self.level3_block8(x7)
        closest_captions_tuple = self.matcher.find_closest_caption(query_image_path, n=1)
        
        x_prompt = self.prompt(x_down3,closest_captions_tuple)
        # print(x8.shape)
        
        x_level3_mix = self.mix1(x_prompt, x8)

        x_up1 = self.up1(x_level3_mix)
        x_up1 = self.up_level2_block1(x_up1)
        x_up1 = self.up_level2_block2(x_up1)
        x_up1 = self.up_level2_block3(x_up1)
        x_up1 = self.up_level2_block4(x_up1)

        x_level2_mix = self.mix2(x_down2, x_up1)
        x_up2 = self.up2(x_level2_mix)
        x_up2 = self.up_level1_block1(x_up2)
        x_up2 = self.up_level1_block2(x_up2)
        x_up2 = self.up_level1_block3(x_up2)
        x_up2 = self.up_level1_block4(x_up2)
        out = self.up3(x_up2)

        return out

LOSS

In [13]:
import torch.nn as nn
import torch
from torch.nn import functional as F
import torch.nn.functional as fnn
from torch.autograd import Variable
import numpy as np
from torchvision import models

class Vgg19(torch.nn.Module):
    def __init__(self, requires_grad=False):
        super(Vgg19, self).__init__()
        vgg_pretrained_features = models.vgg19(pretrained=True).features
        self.slice1 = torch.nn.Sequential()
        self.slice2 = torch.nn.Sequential()
        self.slice3 = torch.nn.Sequential()
        self.slice4 = torch.nn.Sequential()
        self.slice5 = torch.nn.Sequential()
        for x in range(2):
            self.slice1.add_module(str(x), vgg_pretrained_features[x])
        for x in range(2, 7):
            self.slice2.add_module(str(x), vgg_pretrained_features[x])
        for x in range(7, 12):
            self.slice3.add_module(str(x), vgg_pretrained_features[x])
        for x in range(12, 21):
            self.slice4.add_module(str(x), vgg_pretrained_features[x])
        for x in range(21, 30):
            self.slice5.add_module(str(x), vgg_pretrained_features[x])
        if not requires_grad:
            for param in self.parameters():
                param.requires_grad = False

    def forward(self, X):
        h_relu1 = self.slice1(X)
        h_relu2 = self.slice2(h_relu1) 
        h_relu3 = self.slice3(h_relu2)
        h_relu4 = self.slice4(h_relu3)
        h_relu5 = self.slice5(h_relu4) 
        return [h_relu1, h_relu2, h_relu3, h_relu4, h_relu5]

class ContrastLoss(nn.Module):
    def __init__(self, ablation=False):

        super(ContrastLoss, self).__init__()
        self.vgg = Vgg19().cuda()
        self.l1 = nn.L1Loss()
        self.weights = [1.0/32, 1.0/16, 1.0/8, 1.0/4, 1.0]
        self.ab = ablation

    def forward(self, a, p, n):
        a_vgg, p_vgg, n_vgg = self.vgg(a), self.vgg(p), self.vgg(n)
        loss = 0

        d_ap, d_an = 0, 0
        for i in range(len(a_vgg)):
            d_ap = self.l1(a_vgg[i], p_vgg[i].detach())
            if not self.ab:
                d_an = self.l1(a_vgg[i], n_vgg[i].detach())
                contrastive = d_ap / (d_an + 1e-7)
            else:
                contrastive = d_ap

            loss += self.weights[i] * contrastive
        return loss

Original loss functions

In [14]:


def edge_loss(out, target, cuda=True):
	x_filter = np.array([[[1, 0, -1], [2, 0, -2], [1, 0, -1]],[[1, 0, -1], [2, 0, -2], [1, 0, -1]],[[1, 0, -1], [2, 0, -2], [1, 0, -1]]])
	y_filter = np.array([[[1, 2, 1], [0, 0, 0], [-1, -2, -1]],[[1, 2, 1], [0, 0, 0], [-1, -2, -1]],[[1, 2, 1], [0, 0, 0], [-1, -2, -1]]])
	convx = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1, bias=False)
	convy = nn.Conv2d(3, 3, kernel_size=3 , stride=1, padding=1, bias=False)
	weights_x = torch.from_numpy(x_filter).float().unsqueeze(0)
	weights_y = torch.from_numpy(y_filter).float().unsqueeze(0)
    
	if cuda:
		weights_x = weights_x.cuda()
		weights_y = weights_y.cuda()

	convx.weight = nn.Parameter(weights_x)
	convy.weight = nn.Parameter(weights_y)

	g1_x = convx(out)
	g2_x = convx(target)
	g1_y = convy(out)
	g2_y = convy(target)

	g_1 = torch.sqrt(torch.pow(g1_x, 2) + torch.pow(g1_y, 2))
	g_2 = torch.sqrt(torch.pow(g2_x, 2) + torch.pow(g2_y, 2))

	return torch.mean((g_1 - g_2).pow(2))


class lossfunc(nn.Module):
    def __init__(self):
        super(lossfunc, self).__init__()
        self.l1loss = nn.L1Loss()
        self.max_val = 1
        self.window_size = 11
        
        
    def forward(self, out1, tar):
        s1loss = self.l1loss(out1, tar)
        
        o1_fft = torch.fft.fft2(out1)
        t1_fft = torch.fft.fft2(tar)

        edge1loss = edge_loss(out1, tar)
        
        f1loss = self.l1loss(torch.real(o1_fft), torch.real(t1_fft)) + self.l1loss(torch.imag(o1_fft), torch.imag(t1_fft))
        
        # loss_preres = self.l1loss(ref, tar) + 0.1* ssim_loss
        total_loss = s1loss + 0.1*(f1loss) +0.05*(edge1loss)#+ loss_preres #+ 0.1*perceptual_loss
        #total_loss = s1loss #+0.1*f1loss
        return total_loss, s1loss, f1loss, edge1loss


In [20]:
# !jupyter nbconvert --to script "/kaggle/input/option/options-py (1).ipynb" --output "/kaggle/working/options"
# !jupyter nbconvert --to script "/kaggle/input/opt-train/options-train-py (2).ipynb" --output "/kaggle/working/options_train"


In [21]:
import torch
torch.cuda.empty_cache()


In [22]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"


# **Original train.py**

In [15]:
import os, time, math
import numpy as np

import torch
import torch.nn.functional as F
from torch import optim, nn
from torch.backends import cudnn
from torchvision.utils import save_image
from torch.utils.data import DataLoader

# from logger import plot_loss_log, plot_psnr_log
# from metric import psnr, ssim
# from model import DEANet
# from loss import ContrastLoss
# from option_train import opt
# from data.data_loader import TrainDataset, TestDataset


start_time = time.time()
start_time = time.time()
steps = opt.iters_per_epoch * opt.epochs
T = steps


def lr_schedule_cosdecay(t, T, init_lr=opt.start_lr, end_lr=opt.end_lr):
    lr = end_lr + 0.5 * (init_lr - end_lr) * (1 + math.cos(t * math.pi / T))
    return lr


def train(net, loader_train, loader_test, optim, criterion):
    losses = []

    loss_log = {'L1': [], 'CR': [], 'total': []}
    loss_log_tmp = {'L1': [], 'CR': [], 'total': []}
    psnr_log = []

    start_step = 0
    max_ssim = 0
    max_psnr = 0
    ssims = []
    psnrs = []

    loader_train_iter = iter(loader_train)

    for step in range(start_step + 1, steps + 1):
        net.train()
        lr = opt.start_lr
        if not opt.no_lr_sche:
            lr = lr_schedule_cosdecay(step, T)
            for param_group in optim.param_groups:
                param_group["lr"] = lr

        x, y = next(loader_train_iter)
        x = x.to(opt.device)
        y = y.to(opt.device)

        out = net(x)
        if opt.w_loss_L1 > 0:
            loss_L1 = criterion[0](out, y)
        if opt.w_loss_CR > 0:
            loss_CR = criterion[1](out, y, x)
        loss = opt.w_loss_L1 * loss_L1 + opt.w_loss_CR * loss_CR
        loss.backward()
        optim.step()
        optim.zero_grad()
        losses.append(loss.item())
        loss_log_tmp['L1'].append(loss_L1.item())
        loss_log_tmp['CR'].append(loss_CR.item())
        loss_log_tmp['total'].append(loss.item())

        print(
            f'\rloss:{loss.item():.5f} | L1:{loss_L1.item():.5f} | CR:{opt.w_loss_CR * loss_CR.item():.5f} | step :{step}/{steps} | lr :{lr :.7f} | time_used :{(time.time() - start_time) / 60 :.1f}',
            end='', flush=True)

        if step % len(loader_train) == 0:
            loader_train_iter = iter(loader_train)
            for key in loss_log.keys():
                loss_log[key].append(np.average(np.array(loss_log_tmp[key])))
                loss_log_tmp[key] = []
            plot_loss_log(loss_log, int(step / len(loader_train)), opt.saved_plot_dir)
            np.save(os.path.join(opt.saved_data_dir, 'losses.npy'), losses)
        if (step % opt.iters_per_epoch == 0 and step <= opt.finer_eval_step) or (step > opt.finer_eval_step and (step - opt.finer_eval_step) % (5 * len(loader_train)) == 0):
            if step > opt.finer_eval_step:
                epoch = opt.finer_eval_step // opt.iters_per_epoch + (step - opt.finer_eval_step) // (5 * len(loader_train))
            else:
                epoch = int(step / opt.iters_per_epoch)
            with torch.no_grad():
                ssim_eval, psnr_eval = test(net, loader_test)

            log = f'\nstep :{step} | epoch: {epoch} | ssim:{ssim_eval:.4f}| psnr:{psnr_eval:.4f}'
            print(log)
            with open(os.path.join(opt.saved_data_dir, 'log.txt'), 'a') as f:
                f.write(log + '\n')

            ssims.append(ssim_eval)
            psnrs.append(psnr_eval)
            psnr_log.append(psnr_eval)
            plot_psnr_log(psnr_log, epoch, opt.saved_plot_dir)

            if psnr_eval > max_psnr:
                max_ssim = max(max_ssim, ssim_eval)
                max_psnr = max(max_psnr, psnr_eval)
                print(
                    f'\n model saved at step :{step}| epoch: {epoch} | max_psnr:{max_psnr:.4f}| max_ssim:{max_ssim:.4f}')
                saved_best_model_path = os.path.join(opt.saved_model_dir, 'best.pk')
                torch.save({
                    'epoch': epoch,
                    'step': step,
                    'max_psnr': max_psnr,
                    'max_ssim': max_ssim,
                    'ssims': ssims,
                    'psnrs': psnrs,
                    'losses': losses,
                    'model': net.state_dict(),
                    'optimizer': optim.state_dict()
                }, saved_best_model_path)
            saved_single_model_path = os.path.join(opt.saved_model_dir, str(epoch) + '.pk')
            torch.save({
                'epoch': epoch,
                'step': step,
                'max_psnr': max_psnr,
                'max_ssim': max_ssim,
                'ssims': ssims,
                'psnrs': psnrs,
                'losses': losses,
                'model': net.state_dict(),
                'optimizer': optim.state_dict()
            }, saved_single_model_path)
            loader_train_iter = iter(loader_train)
            np.save(os.path.join(opt.saved_data_dir, 'ssims.npy'), ssims)
            np.save(os.path.join(opt.saved_data_dir, 'psnrs.npy'), psnrs)

def pad_img(x, patch_size):
    _, _, h, w = x.size()
    mod_pad_h = (patch_size - h % patch_size) % patch_size
    mod_pad_w = (patch_size - w % patch_size) % patch_size
    x = F.pad(x, (0, mod_pad_w, 0, mod_pad_h), 'reflect')
    return x

def test(net, loader_test):
    net.eval()
    torch.cuda.empty_cache()
    ssims = []
    psnrs = []

    for i, (inputs, targets, hazy_name) in enumerate(loader_test):
        inputs = inputs.to(opt.device)
        targets = targets.to(opt.device)
        with torch.no_grad():
            H, W = inputs.shape[2:]
            inputs = pad_img(inputs, 4)
            pred = net(inputs).clamp(0, 1)
            pred = pred[:, :, :H, :W]
            # save_path = os.path.join(opt.saved_infer_dir, hazy_name[0])
            # save_image(pred, save_path)
        ssim_tmp = ssim(pred, targets).item()
        psnr_tmp = psnr(pred, targets)
        ssims.append(ssim_tmp)
        psnrs.append(psnr_tmp)

    return np.mean(ssims), np.mean(psnrs)


def set_seed_torch(seed=2018):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True


if __name__ == "__main__":

    set_seed_torch(666)

    train_dir = '/kaggle/input/nh-haze/NH-HAZE'
    train_set = TrainDataset(os.path.join(train_dir, 'NH-hazy'), os.path.join(train_dir, 'NH-GT'))
    test_dir = '/kaggle/input/nh-haze/NH-HAZE'
    test_set = TestDataset(os.path.join(test_dir, 'NH-hazy'), os.path.join(test_dir, 'NH-GT'))
    loader_train = DataLoader(dataset=train_set, batch_size=16, shuffle=True, num_workers=12)
    loader_test = DataLoader(dataset=test_set, batch_size=1, shuffle=False, num_workers=4)

    net = DEANet(base_dim=32)
    net = net.to(opt.device)

    epoch_size = len(loader_train)
    print("epoch_size: ", epoch_size)
    if opt.device == 'cuda':
        net = torch.nn.DataParallel(net)
        cudnn.benchmark = True

    pytorch_total_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print("Total_params: ==> {}".format(pytorch_total_params))

    criterion = []
    criterion.append(nn.L1Loss().to(opt.device))
    criterion.append(ContrastLoss(ablation=False))

    optimizer = optim.Adam(params=filter(lambda x: x.requires_grad, net.parameters()), lr=opt.start_lr, betas=(0.9, 0.999),
                           eps=1e-08)
    optimizer.zero_grad()
    train(net, loader_train, loader_test, optimizer, criterion)

ValueError: invalid literal for int() with base 10: '21_hazy'

In [16]:
import os, time, math
import numpy as np

import torch
import torch.nn.functional as F
from torch import optim, nn
from torch.backends import cudnn
from torchvision.utils import save_image
from torch.utils.data import DataLoader

# from logger import plot_loss_log, plot_psnr_log
# from metric import psnr, ssim
# from model import DEANet
# from loss import ContrastLoss
# from option_train import opt
# from data.data_loader import TrainDataset, TestDataset


start_time = time.time()
start_time = time.time()
steps = opt.iters_per_epoch * opt.epochs
T = steps


def lr_schedule_cosdecay(t, T, init_lr=opt.start_lr, end_lr=opt.end_lr):
    lr = end_lr + 0.5 * (init_lr - end_lr) * (1 + math.cos(t * math.pi / T))
    return lr


def train(net, loader_train, loader_test, optim, criterion):
    losses = []

    loss_log = {'L1': [], 'CR': [], 'total': []}
    loss_log_tmp = {'L1': [], 'CR': [], 'total': []}
    psnr_log = []

    start_step = 0
    max_ssim = 0
    max_psnr = 0
    ssims = []
    psnrs = []

    loader_train_iter = iter(loader_train)

    for step in range(start_step + 1, steps + 1):
        net.train()
        lr = opt.start_lr
        if not opt.no_lr_sche:
            lr = lr_schedule_cosdecay(step, T)
            for param_group in optim.param_groups:
                param_group["lr"] = lr

        x, y = next(loader_train_iter)
        x = x.to(opt.device)
        y = y.to(opt.device)

        out = net(x)
        if opt.w_loss_L1 > 0:
            loss_L1 = criterion[0](out, y)
        if opt.w_loss_CR > 0:
            loss_CR = criterion[1](out, y, x)
        loss = opt.w_loss_L1 * loss_L1 + opt.w_loss_CR * loss_CR
        loss.backward()
        optim.step()
        optim.zero_grad()
        losses.append(loss.item())
        loss_log_tmp['L1'].append(loss_L1.item())
        loss_log_tmp['CR'].append(loss_CR.item())
        loss_log_tmp['total'].append(loss.item())

        print(
            f'\rloss:{loss.item():.5f} | L1:{loss_L1.item():.5f} | CR:{opt.w_loss_CR * loss_CR.item():.5f} | step :{step}/{steps} | lr :{lr :.7f} | time_used :{(time.time() - start_time) / 60 :.1f}',
            end='', flush=True)

        if step % len(loader_train) == 0:
            loader_train_iter = iter(loader_train)
            for key in loss_log.keys():
                loss_log[key].append(np.average(np.array(loss_log_tmp[key])))
                loss_log_tmp[key] = []
            plot_loss_log(loss_log, int(step / len(loader_train)), opt.saved_plot_dir)
            np.save(os.path.join(opt.saved_data_dir, 'losses.npy'), losses)
        if (step % opt.iters_per_epoch == 0 and step <= opt.finer_eval_step) or (step > opt.finer_eval_step and (step - opt.finer_eval_step) % (5 * len(loader_train)) == 0):
            if step > opt.finer_eval_step:
                epoch = opt.finer_eval_step // opt.iters_per_epoch + (step - opt.finer_eval_step) // (5 * len(loader_train))
            else:
                epoch = int(step / opt.iters_per_epoch)
            with torch.no_grad():
                ssim_eval, psnr_eval = test(net, loader_test)

            log = f'\nstep :{step} | epoch: {epoch} | ssim:{ssim_eval:.4f}| psnr:{psnr_eval:.4f}'
            print(log)
            with open(os.path.join(opt.saved_data_dir, 'log.txt'), 'a') as f:
                f.write(log + '\n')

            ssims.append(ssim_eval)
            psnrs.append(psnr_eval)
            psnr_log.append(psnr_eval)
            plot_psnr_log(psnr_log, epoch, opt.saved_plot_dir)

            if psnr_eval > max_psnr:
                max_ssim = max(max_ssim, ssim_eval)
                max_psnr = max(max_psnr, psnr_eval)
                print(
                    f'\n model saved at step :{step}| epoch: {epoch} | max_psnr:{max_psnr:.4f}| max_ssim:{max_ssim:.4f}')
                saved_best_model_path = os.path.join(opt.saved_model_dir, 'best.pk')
                torch.save({
                    'epoch': epoch,
                    'step': step,
                    'max_psnr': max_psnr,
                    'max_ssim': max_ssim,
                    'ssims': ssims,
                    'psnrs': psnrs,
                    'losses': losses,
                    'model': net.state_dict(),
                    'optimizer': optim.state_dict()
                }, saved_best_model_path)
            saved_single_model_path = os.path.join(opt.saved_model_dir, str(epoch) + '.pk')
            torch.save({
                'epoch': epoch,
                'step': step,
                'max_psnr': max_psnr,
                'max_ssim': max_ssim,
                'ssims': ssims,
                'psnrs': psnrs,
                'losses': losses,
                'model': net.state_dict(),
                'optimizer': optim.state_dict()
            }, saved_single_model_path)
            loader_train_iter = iter(loader_train)
            np.save(os.path.join(opt.saved_data_dir, 'ssims.npy'), ssims)
            np.save(os.path.join(opt.saved_data_dir, 'psnrs.npy'), psnrs)

def pad_img(x, patch_size):
    _, _, h, w = x.size()
    mod_pad_h = (patch_size - h % patch_size) % patch_size
    mod_pad_w = (patch_size - w % patch_size) % patch_size
    x = F.pad(x, (0, mod_pad_w, 0, mod_pad_h), 'reflect')
    return x

def test(net, loader_test):
    net.eval()
    torch.cuda.empty_cache()
    ssims = []
    psnrs = []

    for i, (inputs, targets, hazy_name) in enumerate(loader_test):
        inputs = inputs.to(opt.device)
        targets = targets.to(opt.device)
        with torch.no_grad():
            H, W = inputs.shape[2:]
            inputs = pad_img(inputs, 4)
            pred = net(inputs).clamp(0, 1)
            pred = pred[:, :, :H, :W]
            # save_path = os.path.join(opt.saved_infer_dir, hazy_name[0])
            # save_image(pred, save_path)
        ssim_tmp = ssim(pred, targets).item()
        psnr_tmp = psnr(pred, targets)
        ssims.append(ssim_tmp)
        psnrs.append(psnr_tmp)

    return np.mean(ssims), np.mean(psnrs)


def set_seed_torch(seed=2018):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True


if __name__ == "__main__":

    set_seed_torch(666)

    train_dir = '/kaggle/input/rain-100l/Rain100L'
    train_set = TrainDataset(os.path.join(train_dir, 'input'), os.path.join(train_dir, 'target'))
    test_dir = '/kaggle/input/rain-100l/Rain100L'
    test_set = TestDataset(os.path.join(test_dir, 'input'), os.path.join(test_dir, 'target'))
    loader_train = DataLoader(dataset=train_set, batch_size=16, shuffle=True, num_workers=12)
    loader_test = DataLoader(dataset=test_set, batch_size=1, shuffle=False, num_workers=4)

    net = DEANet(base_dim=32)
    net = net.to(opt.device)

    epoch_size = len(loader_train)
    print("epoch_size: ", epoch_size)
    if opt.device == 'cuda':
        net = torch.nn.DataParallel(net)
        cudnn.benchmark = True

    pytorch_total_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print("Total_params: ==> {}".format(pytorch_total_params))

    criterion = []
    criterion.append(nn.L1Loss().to(opt.device))
    criterion.append(ContrastLoss(ablation=False))

    optimizer = optim.Adam(params=filter(lambda x: x.requires_grad, net.parameters()), lr=opt.start_lr, betas=(0.9, 0.999),
                           eps=1e-08)
    optimizer.zero_grad()
    train(net, loader_train, loader_test, optimizer, criterion)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


epoch_size:  5
Total_params: ==> 7789313


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 227MB/s]  
<ipython-input-4-ffa8c30fcdfc>:19: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create te

loss:0.25750 | L1:0.11112 | CR:0.14638 | step :10/50 | lr :0.0003619 | time_used :0.6

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


...

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


loss:0.20478 | L1:0.06478 | CR:0.14000 | step :45/50 | lr :0.0000108 | time_used :1.4

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


loss:0.20213 | L1:0.06384 | CR:0.13829 | step :50/50 | lr :0.0000010 | time_used :1.5

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [60]:

import os, time, math
import numpy as np
# import opt
import torch
import torch.nn.functional as F
from torch import optim, nn
from torch.backends import cudnn
from torchvision.utils import save_image
from torch.utils.data import DataLoader
# import TrainOptions as opt
# from logger import plot_loss_log, plot_psnr_log
# from metric import psnr, ssim
# from model import DEANet
# from loss import ContrastLoss
# from options_train import opt
# from data.data_loader import TrainDataset, TestDataset


start_time = time.time()
start_time = time.time()
steps = opt.iters_per_epoch * opt.epochs
T = steps


def lr_schedule_cosdecay(t, T, init_lr=opt.start_lr, end_lr=opt.end_lr):
    lr = end_lr + 0.5 * (init_lr - end_lr) * (1 + math.cos(t * math.pi / T))
    return lr


def train(net, loader_train, loader_test, optim, criterion):
    losses = []

    # loss_log = {'L1': [], 'CR': [], 'total': []}
    loss_log = {'s1': [], 'f1': [], 'edge': [], 'total':[]}
    # loss_log_tmp = {'L1': [], 'CR': [], 'total': []}
    loss_log_tmp ={'s1': [], 'f1': [], 'edge': [], 'total':[]}
    psnr_log = []

    start_step = 0
    max_ssim = 0
    max_psnr = 0
    ssims = []
    psnrs = []

    loader_train_iter = iter(loader_train)

    for step in range(start_step + 1, steps + 1):
        net.train()
        lr = opt.start_lr
        if not opt.no_lr_sche:
            lr = lr_schedule_cosdecay(step, T)
            for param_group in optim.param_groups:
                param_group["lr"] = lr

        x, y ,query_image_path= next(loader_train_iter)
        x = x.to(opt.device)
        y = y.to(opt.device)
        # print(query_image_path)
        # query_image_path = tuple(tensor.to(device) for tensor in query_image_path)
        # print(x.shape,y,query_image_path)
        out = net(x,query_image_path)
        # if opt.w_loss_L1 > 0:
        #     loss_L1 = criterion[0](out, y)
        # if opt.w_loss_CR > 0:
        #     loss_CR = criterion[1](out, y, x)
        # loss = opt.w_loss_L1 * loss_L1 + opt.w_loss_CR * loss_CR

        total_loss, s1loss, f1loss, edge1loss = criterion(out, y)
        loss= total_loss
        loss.backward()
        optim.step()
        optim.zero_grad()
        losses.append(loss.item())
        loss_log_tmp['s1'].append(s1loss.item())
        loss_log_tmp['f1'].append(f1loss.item())
        loss_log_tmp['edge'].append(edge1loss.item())
        loss_log_tmp['total'].append(total_loss.item())

        print(
            f'\rloss:{loss.item():.5f} | L1:{s1loss.item():.5f} | Freq:{0.1 * f1loss.item():.5f} | Edge:{0.05 * edge1loss.item():.5f} | step:{step}/{steps} | lr:{lr:.7f} | time_used:{(time.time() - start_time) / 60:.1f} min',
            end='', flush=True
        )


        if step % len(loader_train) == 0:
            loader_train_iter = iter(loader_train)
            for key in loss_log.keys():
                loss_log[key].append(np.average(np.array(loss_log_tmp[key])))
                loss_log_tmp[key] = []
            plot_loss_log(loss_log, int(step / len(loader_train)), opt.saved_plot_dir)
            np.save(os.path.join(opt.saved_data_dir, 'losses.npy'), losses)
        if (step % opt.iters_per_epoch == 0 and step <= opt.finer_eval_step) or (step > opt.finer_eval_step and (step - opt.finer_eval_step) % (5 * len(loader_train)) == 0):
            if step > opt.finer_eval_step:
                epoch = opt.finer_eval_step // opt.iters_per_epoch + (step - opt.finer_eval_step) // (5 * len(loader_train))
            else:
                epoch = int(step / opt.iters_per_epoch)
            with torch.no_grad():
                ssim_eval, psnr_eval = test(net, loader_test)

            log = f'\nstep :{step} | epoch: {epoch} | ssim:{ssim_eval:.4f}| psnr:{psnr_eval:.4f}'
            print(log)
            with open(os.path.join(opt.saved_data_dir, 'log.txt'), 'a') as f:
                f.write(log + '\n')

            ssims.append(ssim_eval)
            psnrs.append(psnr_eval)
            psnr_log.append(psnr_eval)
            plot_psnr_log(psnr_log, epoch, opt.saved_plot_dir)

            if psnr_eval > max_psnr:
                max_ssim = max(max_ssim, ssim_eval)
                max_psnr = max(max_psnr, psnr_eval)
                print(
                    f'\n model saved at step :{step}| epoch: {epoch} | max_psnr:{max_psnr:.4f}| max_ssim:{max_ssim:.4f}')
                saved_best_model_path = os.path.join(opt.saved_model_dir, 'best.pk')
                torch.save({
                    'epoch': epoch,
                    'step': step,
                    'max_psnr': max_psnr,
                    'max_ssim': max_ssim,
                    'ssims': ssims,
                    'psnrs': psnrs,
                    'losses': losses,
                    'model': net.state_dict(),
                    'optimizer': optim.state_dict()
                }, saved_best_model_path)
            saved_single_model_path = os.path.join(opt.saved_model_dir, str(epoch) + '.pk')
            torch.save({
                'epoch': epoch,
                'step': step,
                'max_psnr': max_psnr,
                'max_ssim': max_ssim,
                'ssims': ssims,
                'psnrs': psnrs,
                'losses': losses,
                'model': net.state_dict(),
                'optimizer': optim.state_dict()
            }, saved_single_model_path)
            loader_train_iter = iter(loader_train)
            np.save(os.path.join(opt.saved_data_dir, 'ssims.npy'), ssims)
            np.save(os.path.join(opt.saved_data_dir, 'psnrs.npy'), psnrs)

def pad_img(x, patch_size):
    _, _, h, w = x.size()
    mod_pad_h = (patch_size - h % patch_size) % patch_size
    mod_pad_w = (patch_size - w % patch_size) % patch_size
    x = F.pad(x, (0, mod_pad_w, 0, mod_pad_h), 'reflect')
    return x

def test(net, loader_test):
    net.eval()
    torch.cuda.empty_cache()
    ssims = []
    psnrs = []

    for i, (degraded,clear, degraded_image_path) in enumerate(loader_test):
        degraded = degraded.to(opt.device)
        clear = clear.to(opt.device)
        with torch.no_grad():
            H, W = degraded.shape[2:]
            degraded = pad_img(degraded, 4)
            pred = net(degraded, degraded_image_path).clamp(0, 1)
            pred = pred[:, :, :H, :W]
            # save_path = os.path.join(opt.saved_infer_dir, hazy_name[0])
            # save_image(pred, save_path)
        ssim_tmp = ssim(pred, clear).item()
        psnr_tmp = psnr(pred, clear)
        ssims.append(ssim_tmp)
        psnrs.append(psnr_tmp)

    return np.mean(ssims), np.mean(psnrs)


def set_seed_torch(seed=2018):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True


if __name__ == "__main__":

    set_seed_torch(666)

    train_dir = '/kaggle/input/prompt-tst/test (1)/ITS/test'
    train_set = TrainDataset(os.path.join(train_dir, 'noisy'), os.path.join(train_dir, 'clear'))
    test_dir = '/kaggle/input/prompt-tst/test (1)/ITS/test'
    test_set = TestDataset(os.path.join(test_dir, 'noisy'), os.path.join(test_dir, 'clear'))
    loader_train = DataLoader(dataset=train_set, batch_size=32, shuffle=True, num_workers=0)
    loader_test = DataLoader(dataset=test_set, batch_size=16, shuffle=False, num_workers=0)

    net = DEANet(base_dim=32)
    net = net.to(opt.device)

    epoch_size = len(loader_train)
    print("epoch_size: ", epoch_size)
    if opt.device == 'cuda':
        net = torch.nn.DataParallel(net)
        cudnn.benchmark = True

    pytorch_total_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print("Total_params: ==> {}".format(pytorch_total_params))

    criterion = []
    # criterion.append(nn.L1Loss().to(opt.device))
    # criterion.append(ContrastLoss(ablation=False))


    criterion= lossfunc()

    optimizer = optim.Adam(params=filter(lambda x: x.requires_grad, net.parameters()), lr=opt.start_lr, betas=(0.9, 0.999),
                           eps=1e-08)
    optimizer.zero_grad()

    train(net, loader_train, loader_test, optimizer, criterion)


epoch_size:  1
Total_params: ==> 77388801


<ipython-input-2-eb0738ceb87d>:19: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:78.)
  conv_weight_cd = torch.cuda.FloatTensor(conv_shape[0], conv_shape[1], 3 * 3).fill_(0)


loss:3.09533 | L1:0.47720 | Freq:2.52306 | Edge:0.09506 | step:1/5 | lr:0.0003619 | time_used:0.1 min
step :1 | epoch: 1 | ssim:0.0899| psnr:8.4983

 model saved at step :1| epoch: 1 | max_psnr:8.4983| max_ssim:0.0899
loss:4.73680 | L1:0.55687 | Freq:3.05540 | Edge:1.12454 | step:2/5 | lr:0.0002621 | time_used:0.2 min
step :2 | epoch: 2 | ssim:0.0262| psnr:10.3353

 model saved at step :2| epoch: 2 | max_psnr:10.3353| max_ssim:0.0899
loss:2.48116 | L1:0.21363 | Freq:2.19069 | Edge:0.07685 | step:3/5 | lr:0.0001389 | time_used:0.3 min
step :3 | epoch: 3 | ssim:0.0209| psnr:8.6032
loss:2.49269 | L1:0.23037 | Freq:2.10698 | Edge:0.15534 | step:4/5 | lr:0.0000391 | time_used:0.4 min
step :4 | epoch: 4 | ssim:0.0246| psnr:9.6055
loss:2.46144 | L1:0.19951 | Freq:2.17377 | Edge:0.08816 | step:5/5 | lr:0.0000010 | time_used:0.4 min
step :5 | epoch: 5 | ssim:0.0248| psnr:9.6464


In [61]:
print(1)

1
